In [ ]:
!pip install -q openai pandas numpy tqdm bert-score transformers torch sentence-transformers faiss-cpu

In [ ]:
import os, json, time, hashlib
import numpy as np
import pandas as pd
from getpass import getpass
from tqdm import tqdm

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from bert_score import score as bert_score
from sentence_transformers import SentenceTransformer
import faiss

from openai import AzureOpenAI



# Credentials are supplied through environment variables

In [ ]:
from openai import AzureOpenAI

AZURE_ENDPOINT = os.environ.get("AZURE_OPENAI_ENDPOINT") or os.environ.get("ENDPOINT_URL")
AZURE_API_KEY = os.environ.get("AZURE_OPENAI_API_KEY")
if not AZURE_ENDPOINT or not AZURE_API_KEY:
    raise EnvironmentError("Set AZURE_OPENAI_ENDPOINT and AZURE_OPENAI_API_KEY before running.")
client = AzureOpenAI(
    azure_endpoint=AZURE_ENDPOINT,
    api_key=AZURE_API_KEY,
    api_version=os.environ.get("AZURE_OPENAI_API_VERSION", "2024-12-01-preview"),
)
GPT_MODEL = os.environ.get("AZURE_OPENAI_DEPLOYMENT", "gpt-5-mini")


In [ ]:
# ============================================================
# PATHS
# ============================================================

RAW_DATA_PATH = "data/cmv_comments.csv"

FINAL_META_DIR = "data/final_topic_metas"

TOPIC_META_FILES = {
    "blm": "blm_meta.parquet",
    "guns": "guns_meta.parquet",
    "samesex": "samesex_meta.parquet",
}

TEXT_COL = "comment_text"
STANCE_COL = "stance_label"

BASELINE_CACHE_DIR = "artifacts/SPR_dense_baseline_cache"
os.makedirs(BASELINE_CACHE_DIR, exist_ok=True)

RAW_META_CACHE = os.path.join(BASELINE_CACHE_DIR, "raw_meta_for_SPR.parquet")
RAW_EMB_CACHE = os.path.join(BASELINE_CACHE_DIR, "raw_SPR_embeddings.npy")
RAW_FAISS_CACHE = os.path.join(BASELINE_CACHE_DIR, "raw_SPR_faiss.index")

In [ ]:
# ============================================================
# Topic utilities
# ============================================================

TOPIC_NAMES = {
    "blm": "Black Lives Matter",
    "guns": "Gun control",
    "samesex": "Same-sex marriage legalization",
}

def topic_name(topic):
    return TOPIC_NAMES[topic]

def opposite_stance(stance):
    if stance == "pro":
        return "anti"
    if stance == "anti":
        return "pro"
    return None

In [ ]:
# ============================================================
# Load datasets
# ============================================================

def load_raw_dataset():
    raw = pd.read_csv(RAW_DATA_PATH, encoding="utf-8-sig")

    if "comment_text" not in raw.columns:
        if "message" in raw.columns:
            raw["comment_text"] = raw["message"]
        else:
            raise ValueError(f"Raw dataset needs comment_text or message. Columns: {raw.columns.tolist()}")

    if "topic_title" not in raw.columns:
        raw["topic_title"] = ""

    raw["comment_text"] = raw["comment_text"].fillna("").astype(str)
    raw["topic_title"] = raw["topic_title"].fillna("").astype(str)

    raw = raw[raw["comment_text"].str.len() > 30].copy()
    raw = raw.reset_index(drop=True)

    return raw


def load_final_meta(topic):
    path = os.path.join(FINAL_META_DIR, TOPIC_META_FILES[topic])

    if path.endswith(".parquet"):
        df = pd.read_parquet(path)
    else:
        df = pd.read_csv(path, encoding="utf-8-sig")

    if "comment_text" not in df.columns and "message" in df.columns:
        df["comment_text"] = df["message"]

    df[STANCE_COL] = df[STANCE_COL].astype(str).str.lower().str.strip()
    df = df[df[STANCE_COL].isin(["pro", "anti"])].copy()
    df = df.reset_index(drop=True)

    return df

In [ ]:
def call_gpt(
    prompt,
    system="You are a helpful assistant.",
    max_tokens=1200,
    retries=3,
    sleep_base=3,
):
    for attempt in range(retries):
        try:
            resp = client.chat.completions.create(
                model=GPT_MODEL,
                messages=[
                    {"role": "system", "content": system},
                    {"role": "user", "content": prompt},
                ],
                max_completion_tokens=max_tokens,
            )

            msg = resp.choices[0].message
            txt = (msg.content or "").strip()
            finish_reason = resp.choices[0].finish_reason

            print("finish_reason:", finish_reason)
            print("usage:", getattr(resp, "usage", None))

            if txt:
                return txt

            print("EMPTY GPT OUTPUT, attempt:", attempt + 1)

        except Exception as e:
            print("GPT ERROR:", repr(e))

        time.sleep(sleep_base * (attempt + 1))

    return ""

In [ ]:
def gpt_json(prompt, system="Return valid JSON only.", max_tokens=1600):
    txt = call_gpt(
        prompt=prompt,
        system=system + "\nReturn ONLY a valid JSON object. No markdown. No explanation.",
        max_tokens=max_tokens,
    )

    if not txt:
        return {"parse_error": True, "raw": ""}

    try:
        return json.loads(txt)
    except Exception:
        cleaned = txt.replace("```json", "").replace("```", "").strip()
        start = cleaned.find("{")
        end = cleaned.rfind("}")

        if start != -1 and end != -1 and end > start:
            try:
                return json.loads(cleaned[start:end + 1])
            except Exception:
                pass

    print("JSON parse failed. Raw text:", txt[:500])
    return {"parse_error": True, "raw": txt}

In [ ]:
# ============================================================
# Build/load SPR-style dense retrieval index
# ============================================================

DENSE_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
dense_model = SentenceTransformer(DENSE_MODEL_NAME)

def build_or_load_SPR_dense_index(force_rebuild=False):
    """
    Adaptation of SPR:
    Treat raw Reddit comments as a large persona memory and retrieve by semantic similarity.

    This baseline intentionally does NOT use:
    - topic extraction from our method
    - stance labels
    - NLI
    - GPT relevance filtering
    - balanced repositories
    - diversity selection
    - final 300 datasets
    """

    if (
        not force_rebuild
        and os.path.exists(RAW_META_CACHE)
        and os.path.exists(RAW_EMB_CACHE)
        and os.path.exists(RAW_FAISS_CACHE)
    ):
        print("Loading cached SPR-style dense retrieval artifacts...")
        raw_meta = pd.read_parquet(RAW_META_CACHE)
        embs = np.load(RAW_EMB_CACHE, mmap_mode="r")
        index = faiss.read_index(RAW_FAISS_CACHE)
        print("Loaded:", len(raw_meta), embs.shape, index.ntotal)
        return raw_meta, embs, index

    print("Building SPR-style dense retrieval index from raw dataset...")

    raw_meta = load_raw_dataset()

    keep_cols = []
    for c in ["comment_id", "author", "comment_text", "topic_title", "thread_id", "delta"]:
        if c in raw_meta.columns:
            keep_cols.append(c)

    raw_meta = raw_meta[keep_cols].copy()
    raw_meta = raw_meta.reset_index(drop=True)
    raw_meta.to_parquet(RAW_META_CACHE, index=False)

    texts = raw_meta["comment_text"].fillna("").astype(str).tolist()

    embs = dense_model.encode(
        texts,
        batch_size=256,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype("float32")

    np.save(RAW_EMB_CACHE, embs)

    dim = embs.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(embs)

    faiss.write_index(index, RAW_FAISS_CACHE)

    print("Built index:", index.ntotal)

    return raw_meta, embs, index


raw_SPR_meta, raw_SPR_embs, raw_SPR_index = build_or_load_SPR_dense_index(
    force_rebuild=False
)

In [ ]:
def SPR_retrieve_candidates(user_opinion, topic, top_k_retrieve=60):
    query_text = f"{topic_name(topic)}. {user_opinion}"

    query_emb = dense_model.encode(
        [query_text],
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype("float32")

    scores, ids = raw_SPR_index.search(query_emb, top_k_retrieve)

    retrieved = []

    for rank, idx in enumerate(ids[0]):
        idx = int(idx)
        row = raw_SPR_meta.iloc[idx].to_dict()

        retrieved.append({
            "rank": rank + 1,
            "similarity": float(scores[0][rank]),
            "text": str(row.get("comment_text", "")),
            "topic_title": str(row.get("topic_title", "")),
            "raw_index": idx,
        })

    return retrieved

In [ ]:
def safe_json_loads(txt):
    txt = (txt or "").strip()
    if not txt:
        return {"parse_error": True, "raw": ""}

    try:
        return json.loads(txt)
    except Exception:
        cleaned = txt.replace("```json", "").replace("```", "").strip()
        start = cleaned.find("{")
        end = cleaned.rfind("}")
        if start != -1 and end != -1 and end > start:
            try:
                return json.loads(cleaned[start:end + 1])
            except Exception:
                pass

    return {"parse_error": True, "raw": txt}


def call_gpt_quiet(prompt, system="You are a helpful assistant.", max_tokens=1800, retries=3):
    for attempt in range(retries):
        try:
            resp = client.chat.completions.create(
                model=GPT_MODEL,
                messages=[
                    {"role": "system", "content": system},
                    {"role": "user", "content": prompt},
                ],
                max_completion_tokens=max_tokens,
            )
            txt = (resp.choices[0].message.content or "").strip()
            if txt:
                return txt
        except Exception as e:
            print("GPT error:", repr(e))

        time.sleep(2 * (attempt + 1))

    return ""


def gpt_json_quiet(prompt, system="Return valid JSON only.", max_tokens=2200):
    txt = call_gpt_quiet(
        prompt=prompt,
        system=system + "\nReturn ONLY valid JSON. No markdown. No explanation.",
        max_tokens=max_tokens,
    )
    return safe_json_loads(txt)

In [ ]:
PERSONADB_DIR = "artifacts/personadb_offline_baseline"
os.makedirs(PERSONADB_DIR, exist_ok=True)

PERSONADB_PARQUET = os.path.join(PERSONADB_DIR, "personadb_refined_items.parquet")
PERSONADB_EMBS = os.path.join(PERSONADB_DIR, "personadb_refined_embeddings.npy")
PERSONADB_FAISS = os.path.join(PERSONADB_DIR, "personadb_refined_faiss.index")

DENSE_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
dense_model = SentenceTransformer(DENSE_MODEL_NAME)

In [ ]:
def make_personadb_user_groups(
    raw_df,
    user_col="author",
    text_col="comment_text",
    min_items_per_user=3,
    max_items_per_user=20,
    max_users=500,
    random_state=42,
):
    """
    Persona-DB assumes user histories.
    In your Reddit dataset, we approximate this by grouping comments by author.
    """

    df = raw_df.copy()

    if user_col not in df.columns:
        df[user_col] = "unknown_user"

    df[text_col] = df[text_col].fillna("").astype(str)
    df = df[df[text_col].str.len() > 30].copy()

    counts = df[user_col].value_counts()
    valid_users = counts[counts >= min_items_per_user].index.tolist()

    if max_users is not None:
        rng = np.random.default_rng(random_state)
        valid_users = list(rng.choice(valid_users, size=min(max_users, len(valid_users)), replace=False))

    groups = []

    for uid in valid_users:
        user_rows = df[df[user_col] == uid].copy()
        if len(user_rows) > max_items_per_user:
            user_rows = user_rows.sample(max_items_per_user, random_state=random_state)

        history = user_rows[text_col].astype(str).tolist()

        groups.append({
            "user_id": str(uid),
            "history": history,
        })

    return groups

In [ ]:
def personadb_offline_extract_user_database(user_id, history):
    """
    Offline hierarchy construction:
    History -> Distilled Persona -> Induced Persona -> Cache
    """

    compact_history = [
        {"id": str(i), "text": txt[:500]}
        for i, txt in enumerate(history)
        if str(txt).strip()
    ]

    prompt = f"""
You are constructing a Persona-DB style hierarchical persona database.

User id:
{user_id}

User history:
{json.dumps(compact_history, ensure_ascii=False)}

Task:
Create a compact persona database from this user's history.

Return a valid JSON object with this exact schema:
{{
  "distilled_personas": [
    {{
      "key": "short concrete label",
      "description": "specific opinion, attitude, preference, or communication pattern grounded in the user's history",
      "supporting_history_ids": ["0", "2"]
    }}
  ],
  "induced_personas": [
    {{
      "key": "short abstract label",
      "description": "higher-level inferred value, worldview, concern, or generalizable trait",
      "source_distilled_keys": ["key1", "key2"]
    }}
  ],
  "cache_keys": [
    "short reusable category",
    "another reusable category"
  ]
}}

Guidelines:
- Distilled personas should be concrete and grounded.
- Induced personas should be higher-level and generalizable.
- Do not invent unsupported facts.
- Do not create contradiction labels.
- Do not classify pro/anti.
- Return 3-8 distilled personas, 2-6 induced personas, and 3-8 cache keys.
"""

    data = gpt_json_quiet(
        prompt,
        system="You build concise Persona-DB hierarchical user persona databases.",
        max_tokens=2400,
    )

    return data

In [ ]:
def build_personadb_offline(
    raw_df,
    force_rebuild=False,
    max_users=500,
    min_items_per_user=3,
    max_items_per_user=20,
):
    """
    Builds and saves the offline Persona-DB refined database.
    Run this once.
    """

    if (not force_rebuild) and os.path.exists(PERSONADB_PARQUET):
        print("Loading existing Persona-DB refined database...")
        return pd.read_parquet(PERSONADB_PARQUET)

    groups = make_personadb_user_groups(
        raw_df=raw_df,
        user_col="author",
        text_col="comment_text",
        min_items_per_user=min_items_per_user,
        max_items_per_user=max_items_per_user,
        max_users=max_users,
    )

    print("Number of user groups:", len(groups))

    rows = []

    for g in tqdm(groups):
        user_id = g["user_id"]
        history = g["history"]

        data = personadb_offline_extract_user_database(user_id, history)

        if data.get("parse_error"):
            continue

        distilled = data.get("distilled_personas", [])
        induced = data.get("induced_personas", [])
        cache_keys = data.get("cache_keys", [])

        for i, dp in enumerate(distilled):
            rows.append({
                "user_id": user_id,
                "layer": "distilled_persona",
                "key": str(dp.get("key", "")),
                "description": str(dp.get("description", "")),
                "cache_keys": json.dumps(cache_keys, ensure_ascii=False),
                "source": "offline_personadb",
            })

        for i, ip in enumerate(induced):
            rows.append({
                "user_id": user_id,
                "layer": "induced_persona",
                "key": str(ip.get("key", "")),
                "description": str(ip.get("description", "")),
                "cache_keys": json.dumps(cache_keys, ensure_ascii=False),
                "source": "offline_personadb",
            })

    personadb_df = pd.DataFrame(rows)
    personadb_df = personadb_df[personadb_df["description"].astype(str).str.len() > 10].copy()
    personadb_df = personadb_df.reset_index(drop=True)

    personadb_df.to_parquet(PERSONADB_PARQUET, index=False)
    print("Saved:", PERSONADB_PARQUET)
    print("Rows:", len(personadb_df))

    return personadb_df

In [ ]:
# raw_df should be your original 700K Reddit dataframe
# If you already loaded it as raw_SPR_meta, you can use that.
personadb_df = build_personadb_offline(
    raw_df=raw_SPR_meta,
    force_rebuild=False,
    max_users=500,          # increase later if you want stronger baseline
    min_items_per_user=3,
    max_items_per_user=20,
)

In [ ]:
def build_or_load_personadb_index(force_rebuild=False):
    if (
        not force_rebuild
        and os.path.exists(PERSONADB_PARQUET)
        and os.path.exists(PERSONADB_EMBS)
        and os.path.exists(PERSONADB_FAISS)
    ):
        print("Loading Persona-DB index...")
        df = pd.read_parquet(PERSONADB_PARQUET)
        embs = np.load(PERSONADB_EMBS, mmap_mode="r")
        index = faiss.read_index(PERSONADB_FAISS)
        print("Loaded:", len(df), embs.shape, index.ntotal)
        return df, embs, index

    df = pd.read_parquet(PERSONADB_PARQUET)

    texts = (
        df["layer"].astype(str)
        + ": "
        + df["key"].astype(str)
        + ". "
        + df["description"].astype(str)
    ).tolist()

    embs = dense_model.encode(
        texts,
        batch_size=256,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype("float32")

    np.save(PERSONADB_EMBS, embs)

    index = faiss.IndexFlatIP(embs.shape[1])
    index.add(embs)

    faiss.write_index(index, PERSONADB_FAISS)

    print("Saved Persona-DB index.")
    return df, embs, index

In [ ]:
personadb_df, personadb_embs, personadb_index = build_or_load_personadb_index(
    force_rebuild=False
)

In [ ]:
# ============================================================
# SPR baseline 1: retrieval only
# No GPT decision.
# This is the cleanest test of semantic persona retrieval.
# ============================================================

def method_SPR_retrieval_only(
    user_opinion,
    topic,
    expected_user_stance,
    top_k_retrieve=60,
):
    retrieved = SPR_retrieve_candidates(
        user_opinion=user_opinion,
        topic=topic,
        top_k_retrieve=top_k_retrieve,
    )

    if not retrieved:
        return ""

    return retrieved[0]["text"]

In [ ]:
# ============================================================
# SPR baseline 2: retrieval + GPT generation
# Important: GPT is NOT part of the original paper.
# This is only an adaptation for generating a comparable persona output.
# We do NOT give GPT the target opposite stance.
# ============================================================

def method_SPR_retrieval_plus_gpt_no_target_stance(
    user_opinion,
    topic,
    expected_user_stance,
    top_k_retrieve=60,
    top_k_persona=8,
):
    retrieved = SPR_retrieve_candidates(
        user_opinion=user_opinion,
        topic=topic,
        top_k_retrieve=top_k_retrieve,
    )

    persona_evidence = [
        {
            "rank": r["rank"],
            "similarity": r["similarity"],
            "topic_title": r["topic_title"][:150],
            "text": r["text"][:700],
        }
        for r in retrieved[:top_k_persona]
    ]

    prompt = f"""
Topic:
{topic_name(topic)}

User opinion:
{user_opinion}

Retrieved persona evidence from a large raw Reddit dataset:
{json.dumps(persona_evidence, ensure_ascii=False)}

Task:
Write one short Reddit-style persona comment grounded in the retrieved evidence.
Do not force an opposite stance unless the retrieved evidence supports it.
Do not mention retrieval, dataset, or AI.
Write 2-4 sentences only.
"""

    out = call_gpt(
        prompt,
        system="You generate concise persona-grounded Reddit-style comments from retrieved evidence.",
        max_tokens=1200,
    )

    # if not out and persona_evidence:
    #     out = persona_evidence[0]["text"]

    return out

In [ ]:
# GPT-5-mini minimal prompting baselines, sourced from the separate minimal evaluation notebook.
# The original seven method identifiers are retained for downstream analysis.
SYSTEM_PROMPT = "You are a helpful assistant."
FEW_SHOT_DEMOS = """Opinion: School uniforms are good because they reduce social pressure.
Opposing comment: Uniforms do not solve the reasons students compare themselves, and they restrict personal expression.

Opinion: Cities should invest much more money in public transportation.
Opposing comment: More funding alone does not guarantee useful transit, and some projects waste money without improving service.

Opinion: Kids should not be allowed to use social media until they are older.
Opposing comment: A blanket age limit is too restrictive, and families should decide when their children are ready.
"""

def build_prompt(method, opinion):
    if method.startswith('zero_shot_'):
        return f'Write a comment disagreeing with this opinion:\n\n{opinion}'
    if method.startswith('few_shot_'):
        return FEW_SHOT_DEMOS + f'\nOpinion: {opinion}\nOpposing comment:'
    raise ValueError(method)

def _minimal_gpt_response(method, user_opinion, retries=4):
    """Run the exact minimal GPT-5-mini prompt; no topic or stance hints are added."""
    prompt = build_prompt(method, user_opinion)
    error = ""
    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model=GPT_MODEL,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": prompt},
                ],
                max_completion_tokens=1200,
            )
            output = (response.choices[0].message.content or "").strip()
            if output:
                return output
            error = "Empty GPT output (possible reasoning-token exhaustion)"
        except Exception as exc:
            error = repr(exc)
        print("Minimal GPT attempt", attempt + 1, "failed:", error[:200])
        time.sleep(min(30, 2 ** attempt * 3))
    return ""


def method_zero_shot(user_opinion, topic, expected_user_stance):
    return _minimal_gpt_response("zero_shot_gpt_minimal", user_opinion)


def method_few_shot_task_prompting(user_opinion, topic, expected_user_stance):
    return _minimal_gpt_response("few_shot_gpt_minimal", user_opinion)


In [ ]:
# Zero-shot and few-shot methods are defined in the preceding cell.


In [ ]:
# ============================================================
# Partner Matters baseline:
# persona fusion for retrieval-based response selection
# inspired by Gu et al. (SIGIR 2021)
# ============================================================

def cosine(a, b):
    return float(np.dot(a, b))


def partner_matters_retrieve_personas(
    user_opinion,
    topic,
    top_k_personas=8,
):
    """
    Retrieve persona candidates from the raw 700K semantic index.
    This is NOT stance-aware and does NOT use NLI.
    """

    query_text = f"{topic_name(topic)}. {user_opinion}"

    query_emb = dense_model.encode(
        [query_text],
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype("float32")

    scores, ids = raw_SPR_index.search(query_emb, top_k_personas)

    personas = []

    for rank, idx in enumerate(ids[0]):
        idx = int(idx)
        row = raw_SPR_meta.iloc[idx].to_dict()

        personas.append({
            "rank": rank + 1,
            "similarity": float(scores[0][rank]),
            "text": str(row.get("comment_text", ""))[:900],
            "topic_title": str(row.get("topic_title", ""))[:200],
        })

    return personas


def partner_matters_build_response_candidates(
    user_opinion,
    topic,
    expected_user_stance,
    raw_top_k=60,
    candidate_k=12,
):
    """
    Build response candidates from raw semantic retrieval.
    In the original paper, response candidates are given by the response-selection dataset.
    In our adaptation, we use retrieved raw Reddit comments as candidate responses.
    """

    retrieved = SPR_retrieve_candidates(
        user_opinion=user_opinion,
        topic=topic,
        top_k_retrieve=raw_top_k,
    )

    candidates = []

    for r in retrieved[:candidate_k]:
        txt = str(r["text"]).strip()
        if txt:
            candidates.append({
                "text": txt[:900],
                "similarity": float(r["similarity"]),
                "topic_title": str(r.get("topic_title", ""))[:200],
            })

    return candidates

In [ ]:
def method_partner_matters_persona_fusion_response_selection(
    user_opinion,
    topic,
    expected_user_stance,
    top_k_personas=8,
    raw_top_k_candidates=60,
    candidate_k=12,
):
    """
    Adaptation of:
    Gu et al. 2021, Partner Matters!

    Core idea:
    - retrieve persona evidence
    - retrieve candidate responses
    - fuse context/persona/response signals
    - select the response with the best fused score

    No GPT.
    No NLI.
    No stance labels.
    No final 300 repository.
    """

    target_stance = opposite_stance(expected_user_stance)

    # Context embedding
    context_text = f"{topic_name(topic)}. {user_opinion}"
    context_emb = dense_model.encode(
        [context_text],
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype("float32")[0]

    # Retrieve multiple persona candidates
    personas = partner_matters_retrieve_personas(
        user_opinion=user_opinion,
        topic=topic,
        top_k_personas=top_k_personas,
    )

    persona_texts = [p["text"] for p in personas if str(p["text"]).strip()]

    if not persona_texts:
        return ""

    persona_embs = dense_model.encode(
        persona_texts,
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype("float32")

    # Persona fusion: average retrieved persona representations
    fused_persona_emb = persona_embs.mean(axis=0)
    fused_persona_emb = fused_persona_emb / (np.linalg.norm(fused_persona_emb) + 1e-12)

    # Retrieve candidate responses from raw dataset
    response_candidates = partner_matters_build_response_candidates(
        user_opinion=user_opinion,
        topic=topic,
        expected_user_stance=expected_user_stance,
        raw_top_k=raw_top_k_candidates,
        candidate_k=candidate_k,
    )

    if not response_candidates:
        return persona_texts[0]

    response_texts = [c["text"] for c in response_candidates]

    response_embs = dense_model.encode(
        response_texts,
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype("float32")

    scored = []

    for i, c in enumerate(response_candidates):
        r_emb = response_embs[i]

        # context-response matching
        context_response_score = cosine(context_emb, r_emb)

        # persona-response matching
        persona_response_score = cosine(fused_persona_emb, r_emb)

        # context-persona matching
        context_persona_score = cosine(context_emb, fused_persona_emb)

        # Fusion score inspired by context/persona/response interaction
        fusion_score = (
            0.45 * context_response_score
            + 0.45 * persona_response_score
            + 0.10 * context_persona_score
        )

        scored.append({
            "text": c["text"],
            "fusion_score": fusion_score,
            "context_response_score": context_response_score,
            "persona_response_score": persona_response_score,
            "context_persona_score": context_persona_score,
        })

    best = max(scored, key=lambda x: x["fusion_score"])

    return best["text"]

In [ ]:
def coo_retrieve_semantic_candidates(
    user_opinion,
    topic,
    top_k_retrieve=60,
):
    """
    Retrieves semantic candidates from the raw 700K FAISS index.
    This reuses the same raw_SPR_index and raw_SPR_meta
    that were already built for the SPR baseline.
    """

    query_text = f"{topic_name(topic)}. {user_opinion}"

    query_emb = dense_model.encode(
        [query_text],
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype("float32")

    scores, ids = raw_SPR_index.search(query_emb, top_k_retrieve)

    candidates = []

    for rank, idx in enumerate(ids[0]):
        idx = int(idx)
        row = raw_SPR_meta.iloc[idx].to_dict()

        txt = str(row.get("comment_text", "")).strip()

        if not txt:
            continue

        candidates.append({
            "id": str(rank),
            "raw_index": idx,
            "semantic_rank": rank + 1,
            "semantic_similarity": float(scores[0][rank]),
            "topic_title": str(row.get("topic_title", ""))[:200],
            "text": txt[:900],
        })

    return candidates

In [ ]:
def coo_style_rank_by_usefulness(
    user_opinion,
    topic,
    candidates,
    top_k_useful=8,
):
    """
    COO-style LLMtop-K adaptation.

    In the COO paper, the LLM ranks implicit personae/historical opinions
    by usefulness for predicting the user's opinion.

    Here, we adapt that idea:
    The LLM ranks raw retrieved comments by usefulness for constructing
    a relevant persona on the given topic.

    It is intentionally NOT asked to rank by contradiction.
    """

    if not candidates:
        return []

    compact_candidates = [
        {
            "id": c["id"],
            "semantic_rank": c["semantic_rank"],
            "semantic_similarity": round(c["semantic_similarity"], 4),
            "topic_title": c["topic_title"],
            "text": c["text"][:700],
        }
        for c in candidates
    ]

    prompt = f"""
You are ranking candidate persona evidence for a personalized dialogue system.

Topic:
{topic_name(topic)}

User opinion:
{user_opinion}

Candidate comments retrieved from a large raw Reddit dataset:
{json.dumps(compact_candidates, ensure_ascii=False)}

Task:
Rank the candidate comments by how useful they are as persona evidence
for constructing a realistic Reddit-style persona related to the topic.

Important:
- Rank by usefulness, relevance, clarity, and persona richness.
- Do NOT rank by contradiction.
- Do NOT assume the selected persona must oppose the user.
- Ignore irrelevant, unclear, off-topic, or low-quality comments.

Return a valid JSON object with this exact schema:
{{
  "ranked_ids": ["id1", "id2", "id3"],
  "reasons": {{
    "id1": "brief reason",
    "id2": "brief reason"
  }}
}}

Return at most {top_k_useful} ids.
Do not output any text outside the JSON object.
"""

    data = gpt_json(
        prompt,
        system="You are a careful persona-evidence ranker. Return valid JSON only.",
        max_tokens=1200,
    )

    ranked_ids = data.get("ranked_ids", [])

    if not isinstance(ranked_ids, list):
        ranked_ids = []

    ranked_ids = [str(x) for x in ranked_ids]

    by_id = {str(c["id"]): c for c in candidates}

    selected = []

    for rid in ranked_ids:
        if rid in by_id:
            selected.append(by_id[rid])
        if len(selected) >= top_k_useful:
            break

    # fallback: if JSON failed or ids are bad, use semantic top-k
    if not selected:
        selected = candidates[:top_k_useful]

    return selected

In [ ]:
def method_coo_style_usefulness_ranking(
    user_opinion,
    topic,
    expected_user_stance,
    top_k_retrieve=60,
    top_k_useful=8,
):
    """
    COO-style usefulness baseline.

    Pipeline:
    1. Retrieve top-60 semantic candidates from raw 700K dataset.
    2. Ask GPT to rank them by usefulness as persona evidence.
    3. Use top-8 useful candidates.
    4. Generate a persona from those selected candidates.

    This baseline tests:
    Is usefulness-aware persona selection enough,
    without explicit contradiction modeling?
    """

    target_stance = opposite_stance(expected_user_stance)

    candidates = coo_retrieve_semantic_candidates(
        user_opinion=user_opinion,
        topic=topic,
        top_k_retrieve=top_k_retrieve,
    )

    selected = coo_style_rank_by_usefulness(
        user_opinion=user_opinion,
        topic=topic,
        candidates=candidates,
        top_k_useful=top_k_useful,
    )

    selected_compact = [
        {
            "id": c["id"],
            "semantic_rank": c["semantic_rank"],
            "semantic_similarity": round(c["semantic_similarity"], 4),
            "topic_title": c["topic_title"],
            "text": c["text"][:700],
        }
        for c in selected
    ]

    prompt = f"""
You are constructing a Reddit-style persona comment from selected useful persona evidence.

Topic:
{topic_name(topic)}

User opinion:
{user_opinion}

Selected useful persona evidence:
{json.dumps(selected_compact, ensure_ascii=False)}

Task:
Write one short Reddit-style persona comment grounded in the selected evidence.

Important:
- Use the selected evidence for style, topic framing, and persona realism.
- Do not copy the evidence exactly.
- Do not mention the dataset, retrieval, or AI.
- Write 2-4 sentences only.

Note:
This baseline ranks by usefulness, not contradiction.
Therefore, do not force an opposite stance unless the selected evidence clearly supports it.
"""

    out = call_gpt(
        prompt,
        system="You generate concise persona-grounded Reddit-style comments from selected useful evidence.",
        max_tokens=1200,
    )

    # if not out and selected:
    #     out = selected[0]["text"]

    # if not out:
    #     out = ""

    return out

In [ ]:
def personadb_online_retrieve(
    user_opinion,
    topic,
    top_k_items=8,
):
    """
    Online Persona-DB retrieval:
    retrieve from the precomputed refined persona database.
    """

    query = f"{topic_name(topic)}. {user_opinion}"

    q_emb = dense_model.encode(
        [query],
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype("float32")

    scores, ids = personadb_index.search(q_emb, top_k_items)

    retrieved = []

    for rank, idx in enumerate(ids[0]):
        idx = int(idx)
        row = personadb_df.iloc[idx].to_dict()

        retrieved.append({
            "rank": rank + 1,
            "score": float(scores[0][rank]),
            "user_id": str(row.get("user_id", "")),
            "layer": str(row.get("layer", "")),
            "key": str(row.get("key", "")),
            "description": str(row.get("description", "")),
        })

    return retrieved

In [ ]:
def method_personadb_offline_online_baseline(
    user_opinion,
    topic,
    expected_user_stance,
    top_k_items=8,
):
    """
    Fair Persona-DB-style baseline:
    Offline hierarchy already exists.
    Online stage only retrieves refined persona items and generates output.

    No stance filtering.
    No NLI.
    No contradiction reranking.
    No balanced pro/anti repository.
    """

    retrieved_items = personadb_online_retrieve(
        user_opinion=user_opinion,
        topic=topic,
        top_k_items=top_k_items,
    )

    if not retrieved_items:
        return ""

    prompt = f"""
You are generating a Reddit-style persona comment from a precomputed Persona-DB refined persona database.

Topic:
{topic_name(topic)}

User opinion:
{user_opinion}

Retrieved refined persona items:
{json.dumps(retrieved_items, ensure_ascii=False)}

Task:
Write one short Reddit-style persona comment grounded in the retrieved refined persona items.

Important:
- Use the retrieved items for worldview, style, concerns, and topic framing.
- Do not mention Persona-DB, retrieval, dataset, or AI.
- Do not copy the items exactly.
- Write 2-4 sentences only.
- This baseline is not contradiction-aware, so do not force an opposite stance unless the retrieved items naturally support it.
"""

    out = call_gpt_quiet(
        prompt,
        system="You generate concise Reddit-style persona comments from retrieved refined persona items.",
        max_tokens=1000,
    )

    return out

In [ ]:
# ============================================================
# Our method
# ============================================================

def runtime_pool_score(df):
    df = df.copy()

    for col in [
        "gpt_final_score",
        "keep_score",
        "persona_usefulness",
        "coverage_score",
        "uniqueness_score",
        "clarity_score",
        "redundancy_risk",
        "hnsw_similarity",
        "topic_candidate_score",
    ]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0.0)

    score = np.zeros(len(df), dtype=np.float32)

    if "gpt_final_score" in df.columns:
        score += 2.0 * df["gpt_final_score"].values
    if "keep_score" in df.columns:
        score += 1.5 * df["keep_score"].values
    if "persona_usefulness" in df.columns:
        score += 1.2 * df["persona_usefulness"].values
    if "coverage_score" in df.columns:
        score += 1.0 * df["coverage_score"].values
    if "uniqueness_score" in df.columns:
        score += 0.8 * df["uniqueness_score"].values
    if "clarity_score" in df.columns:
        score += 0.5 * df["clarity_score"].values
    if "redundancy_risk" in df.columns:
        score -= 0.8 * df["redundancy_risk"].values
    if "hnsw_similarity" in df.columns:
        score += 0.5 * df["hnsw_similarity"].values
    if "topic_candidate_score" in df.columns:
        score += 0.3 * df["topic_candidate_score"].values

    return score


def method_ours_nli_gpt(
    user_opinion,
    topic,
    expected_user_stance,
    meta,
    pool_size=60,
    k2_short=8,
):
    target_stance = opposite_stance(expected_user_stance)

    pool = meta[meta[STANCE_COL] == target_stance].copy()
    pool = pool.reset_index(drop=True)

    if pool.empty:
        return ""

    pool["_runtime_pool_score"] = runtime_pool_score(pool)
    pool = pool.sort_values("_runtime_pool_score", ascending=False).head(pool_size).copy()

    texts = pool[TEXT_COL].fillna("").astype(str).tolist()

    nli_scores = method_nli_contradiction_scores_batch(
        premise=user_opinion,
        hypotheses=texts,
        batch_size=16,
    )

    pool["nli_score"] = nli_scores
    shortlist = pool.sort_values("nli_score", ascending=False).head(k2_short).copy()

    candidates = []
    for i, row in shortlist.iterrows():
        candidates.append({
            "id": str(i),
            "text": str(row[TEXT_COL])[:700],
            "nli_score": float(row["nli_score"]),
        })

    if not candidates:
        return ""

    prompt = f"""
Topic: {topic_name(topic)}

User opinion:
{user_opinion}

Candidate opposite-stance comments:
{json.dumps(candidates, ensure_ascii=False)}

Select the single best comment for constructing an opposite-opinion persona.
Prefer direct contradiction, topic relevance, clarity, and persona usefulness.

Return a valid JSON object with this exact schema:
{{
  "best_id": "id",
  "reason": "brief reason"
}}
Do not output any text outside the JSON object.
"""

    data = gpt_json(
        prompt,
        system="You are a careful contradiction reranker. Return valid JSON only.",
        max_tokens=1200,
    )

    best_id = str(data.get("best_id", candidates[0]["id"]))
    best = next((c for c in candidates if c["id"] == best_id), candidates[0])

    return best["text"]

In [ ]:
# ============================================================
# NLI models
# IMPORTANT:
# - METHOD NLI is used inside our method for NLI reranking.
# - EVALUATION NLI is used only for the reported NLI metric.
# This avoids evaluating our method with the exact same NLI model
# used inside the method.
# ============================================================

from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"

METHOD_NLI_MODEL_NAME = "MoritzLaurer/multilingual-MiniLMv2-L6-mnli-xnli"
EVAL_NLI_MODEL_NAME = "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"

# Method NLI model: used inside our retrieval/reranking pipeline
method_tokenizer = AutoTokenizer.from_pretrained(METHOD_NLI_MODEL_NAME)
method_nli_model = AutoModelForSequenceClassification.from_pretrained(
    METHOD_NLI_MODEL_NAME
).to(device)
method_nli_model.eval()

# Evaluation NLI model: used only for the reported metric
eval_tokenizer = AutoTokenizer.from_pretrained(EVAL_NLI_MODEL_NAME)
eval_nli_model = AutoModelForSequenceClassification.from_pretrained(
    EVAL_NLI_MODEL_NAME
).to(device)
eval_nli_model.eval()

print("Device:", device)
print("Method NLI model:", METHOD_NLI_MODEL_NAME)
print("Evaluation NLI model:", EVAL_NLI_MODEL_NAME)

In [ ]:
# ============================================================
# Generic NLI contradiction helpers
# ============================================================

def find_contradiction_index(model):
    """
    Finds the contradiction label index for any HuggingFace NLI model.
    """

    id2label = model.config.id2label

    for idx, label in id2label.items():
        if "contradiction" in str(label).lower():
            return int(idx)

    raise ValueError(f"Could not find contradiction label in: {id2label}")


METHOD_CONTRA_IDX = find_contradiction_index(method_nli_model)
EVAL_CONTRA_IDX = find_contradiction_index(eval_nli_model)

print("Method contradiction index:", METHOD_CONTRA_IDX)
print("Evaluation contradiction index:", EVAL_CONTRA_IDX)


@torch.inference_mode()
def get_contradiction_score_with_model(
    premise,
    hypothesis,
    tokenizer,
    model,
    contradiction_idx,
    max_length=512,
):
    """
    Computes contradiction probability for one premise-hypothesis pair.
    """

    if not str(hypothesis).strip():
        return None

    batch = tokenizer(
        str(premise),
        str(hypothesis),
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=max_length,
    )

    batch = {k: v.to(device) for k, v in batch.items()}

    logits = model(**batch).logits
    probs = torch.softmax(logits, dim=-1)[0]

    return float(probs[contradiction_idx].detach().cpu())


@torch.inference_mode()
def get_contradiction_scores_batch_with_model(
    premise,
    hypotheses,
    tokenizer,
    model,
    contradiction_idx,
    batch_size=16,
    max_length=512,
):
    """
    Computes contradiction probabilities for one premise and many hypotheses.
    """

    scores = []

    for i in range(0, len(hypotheses), batch_size):
        batch_h = hypotheses[i:i + batch_size]
        premises = [str(premise)] * len(batch_h)

        batch = tokenizer(
            premises,
            [str(h) for h in batch_h],
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=max_length,
        )

        batch = {k: v.to(device) for k, v in batch.items()}

        logits = model(**batch).logits
        probs = torch.softmax(logits, dim=-1)[:, contradiction_idx]

        scores.extend(probs.detach().cpu().numpy().tolist())

    return np.array(scores, dtype=np.float32)

In [ ]:
# ============================================================
# Method NLI functions
# Used ONLY inside our method for retrieval/reranking
# ============================================================

def method_nli_contradiction_score(premise, hypothesis):
    return get_contradiction_score_with_model(
        premise=premise,
        hypothesis=hypothesis,
        tokenizer=method_tokenizer,
        model=method_nli_model,
        contradiction_idx=METHOD_CONTRA_IDX,
    )


def method_nli_contradiction_scores_batch(premise, hypotheses, batch_size=16):
    return get_contradiction_scores_batch_with_model(
        premise=premise,
        hypotheses=hypotheses,
        tokenizer=method_tokenizer,
        model=method_nli_model,
        contradiction_idx=METHOD_CONTRA_IDX,
        batch_size=batch_size,
    )

In [ ]:
# ============================================================
# Evaluation NLI functions
# Used ONLY for reported NLI metric
# ============================================================

def eval_nli_contradiction_score(premise, hypothesis):
    return get_contradiction_score_with_model(
        premise=premise,
        hypothesis=hypothesis,
        tokenizer=eval_tokenizer,
        model=eval_nli_model,
        contradiction_idx=EVAL_CONTRA_IDX,
    )


def eval_nli_contradiction_scores_batch(premise, hypotheses, batch_size=16):
    return get_contradiction_scores_batch_with_model(
        premise=premise,
        hypotheses=hypotheses,
        tokenizer=eval_tokenizer,
        model=eval_nli_model,
        contradiction_idx=EVAL_CONTRA_IDX,
        batch_size=batch_size,
    )

In [ ]:
# ============================================================
# Metrics
# ============================================================

def compute_bertscore(user_text, candidate_text):
    if not str(candidate_text).strip():
        return {
            "bertscore_precision": None,
            "bertscore_recall": None,
            "bertscore_f1": None,
        }

    try:
        P, R, F1 = bert_score(
            [candidate_text],
            [user_text],
            lang="en",
            model_type="roberta-large",
            verbose=False,
            rescale_with_baseline=False,
            use_fast_tokenizer=True,
        )

        return {
            "bertscore_precision": float(P[0]),
            "bertscore_recall": float(R[0]),
            "bertscore_f1": float(F1[0]),
        }

    except Exception as e:
        print("BERTScore failed:", repr(e))
        return {
            "bertscore_precision": None,
            "bertscore_recall": None,
            "bertscore_f1": None,
        }

# ============================================================
# 3. GPT stance classifier + opposite stance accuracy
# ============================================================

def classify_persona_stance_with_gpt(output_text, topic):
    if not str(output_text).strip():
        return {
            "predicted_persona_stance": "empty",
            "stance_confidence": 0.0,
            "stance_reason": "empty output",
        }

    prompt = f"""
Topic:
{topic_name(topic)}

Persona text:
{output_text}

Classify the persona text's stance toward the topic.

Labels:
- "pro": supports the topic/movement/policy/right.
- "anti": opposes the topic/movement/policy/right.
- "unclear": unclear, mixed, irrelevant, or ambiguous.

Return valid JSON only:
{{
  "stance": "pro",
  "confidence": 0.0,
  "reason": "brief explanation"
}}
"""

    data = gpt_json_quiet(
        prompt,
        system="You are a careful stance classifier. Return valid JSON only.",
        max_tokens=800,
    )

    stance = str(data.get("stance", "unclear")).lower().strip()
    if stance not in ["pro", "anti", "unclear"]:
        stance = "unclear"

    try:
        confidence = float(data.get("confidence", 0.0))
    except Exception:
        confidence = 0.0

    confidence = max(0.0, min(1.0, confidence))

    return {
        "predicted_persona_stance": stance,
        "stance_confidence": confidence,
        "stance_reason": str(data.get("reason", ""))[:300],
    }


def compute_stance_accuracy(output_text, topic, expected_user_stance):
    target = opposite_stance(expected_user_stance)

    stance_info = classify_persona_stance_with_gpt(
        output_text=output_text,
        topic=topic,
    )

    predicted = stance_info["predicted_persona_stance"]

    return {
        **stance_info,
        "target_persona_stance": target,
        "opposite_stance_correct": bool(predicted == target),
        "opposite_stance_accuracy": 1.0 if predicted == target else 0.0,
    }


# ============================================================
# Cosine Similarity Utilities
# ============================================================

from sentence_transformers import SentenceTransformer
import numpy as np

# Use the same embedding model used in the retrieval baselines.
# If you already loaded dense_model earlier, this will reuse it.
try:
    topic_sim_model = dense_model
except NameError:
    topic_sim_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")


def cosine_similarity(vec_a, vec_b):
    """
    General cosine similarity function.

    If vectors are already normalized, this is equivalent to dot product.
    If not, it safely normalizes them.
    """

    vec_a = np.asarray(vec_a, dtype=np.float32)
    vec_b = np.asarray(vec_b, dtype=np.float32)

    denom = (np.linalg.norm(vec_a) * np.linalg.norm(vec_b)) + 1e-12
    return float(np.dot(vec_a, vec_b) / denom)


def compute_topic_cosine_similarity(user_opinion, output_text):
    """
    Evaluation metric:
    Measures whether the produced persona stays semantically/topically
    related to the user's original opinion.

    Higher = more topically similar.
    """

    if not str(output_text).strip():
        return None

    embs = topic_sim_model.encode(
        [str(user_opinion), str(output_text)],
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype("float32")

    # Since normalize_embeddings=True, dot product = cosine similarity.
    return float(np.dot(embs[0], embs[1]))


def evaluate_output(user_opinion, output_text, topic, expected_user_stance):
    valid_output = bool(str(output_text).strip())

    # Evaluation NLI uses microsoft/deberta-large-mnli,
    # not the same NLI model used inside our method.
    nli_score = (
        eval_nli_contradiction_score(user_opinion, output_text)
        if valid_output
        else None
    )

    bs = compute_bertscore(user_opinion, output_text)

    topic_cosine = compute_topic_cosine_similarity(
        user_opinion=user_opinion,
        output_text=output_text,
    )

    stance_metrics = compute_stance_accuracy(
        output_text=output_text,
        topic=topic,
        expected_user_stance=expected_user_stance,
    )

    return {
        "valid_output": valid_output,
        "eval_nli_model": EVAL_NLI_MODEL_NAME,
        "method_nli_model": METHOD_NLI_MODEL_NAME,
        "nli_contradiction": nli_score,
        **bs,
        "topic_cosine_similarity": topic_cosine,
        **stance_metrics,
        "response_words": len(str(output_text).split()) if valid_output else 0,
    }

In [ ]:
# ============================================================
# CELL 35 — FIXED 600-CASE EVALUATION SET
# Same exact evaluation cases used in the Ablation experiment
# 3 topics × 2 stances × 100 cases
# Self-contained: no external test-case file is required
# ============================================================

import pandas as pd

# Frozen evaluation cases: 100 opinions for each topic × stance.
# Do not shuffle, resample, or regenerate these lists.

BLM_PRO = ['Black Lives Matter is important because police violence and racial discrimination are real problems that need public '
 'attention.',
 'I support BLM because Black communities still face systemic racism, and ignoring that only protects the status quo.',
 'BLM is necessary because racial injustice did not disappear, and people need to protest when institutions fail them.',
 'The movement matters because police accountability and racial equality should not be controversial ideas.',
 'I think BLM helped expose problems that many people preferred to ignore, especially around policing and '
 'discrimination.',
 'BLM gives people a way to demand justice when official institutions do not take police violence seriously.',
 'I support BLM because equal treatment under the law is still not experienced equally by everyone.',
 'The protests were justified because people were tired of seeing repeated cases of police brutality without '
 'accountability.',
 'BLM is not about hating anyone; it is about saying that Black lives should be protected and valued equally.',
 'I think the movement is important because racial inequality still affects policing, housing, education, and '
 'opportunity.',
 'BLM brought attention to injustice that many communities have been talking about for decades.',
 'Supporting BLM means supporting accountability, fairness, and equal protection for Black citizens.',
 'I believe BLM is needed because silence about racism allows unfair systems to continue.',
 'The movement helped make police violence and systemic discrimination part of the national conversation.',
 'I support BLM because people should not have to accept discrimination as normal.',
 'BLM is a legitimate response to real cases where Black people were treated unfairly or violently.',
 'I think criticizing BLM ignores the deeper problem of racial injustice that the movement is trying to address.',
 'The slogan Black Lives Matter is not saying other lives do not matter; it is highlighting a group that has been '
 'harmed.',
 'BLM is important because public pressure is often necessary before institutions make meaningful reforms.',
 'I support the movement because everyone should care when a community repeatedly faces unequal treatment.',
 'BLM represents a demand for dignity, fairness, and accountability in policing.',
 'I think the movement gave a voice to people who felt ignored by courts, police departments, and politicians.',
 'BLM is necessary because racism can exist in systems even when individuals deny being racist.',
 'The movement matters because police accountability should be a basic democratic principle.',
 'I support BLM because racial justice is not a radical demand.',
 'BLM helped show that many people experience law enforcement differently depending on race.',
 'I think the movement is justified because injustice should be confronted publicly, not hidden.',
 'BLM is important because communities need protection from both crime and abusive policing.',
 'I support BLM because equality requires recognizing when one group faces specific harms.',
 'The movement is necessary because without protest, many cases of injustice would be forgotten.',
 'I support BLM because racial disparities in access to quality legal representation can affect whether people receive '
 'equal treatment in the justice system.',
 'The movement encouraged conversations about criminal justice reform that had been ignored for many years.',
 'BLM is important because unequal opportunities in education and housing are also part of racial inequality.',
 'I believe racial inequality affects economic opportunities, so BLM is about more than policing.',
 'The movement helped increase public awareness about discrimination in employment and hiring.',
 'Supporting BLM means recognizing that equal legal rights do not always translate into equal opportunities.',
 'I think BLM encouraged governments and institutions to reconsider policies that unintentionally disadvantage '
 'minority communities.',
 'The protests showed that many citizens are willing to participate peacefully to demand institutional reform.',
 'BLM helped encourage discussions about improving relationships between police departments and local communities.',
 'I support BLM because communities deserve transparency when investigating police misconduct.',
 'The movement encouraged many organizations to examine their own diversity and inclusion policies.',
 'BLM highlighted that historical inequalities can continue affecting people even after laws change.',
 'I think addressing racial inequality requires both policing reforms and broader social investment.',
 'Supporting BLM means supporting equal opportunities in education, employment, and justice.',
 'The movement helped many people learn about historical patterns of discrimination that they had never studied.',
 'I support BLM because independent investigations of police misconduct are necessary when departments cannot credibly '
 'investigate themselves.',
 'BLM has helped make body-camera policies and transparency in policing part of a broader public discussion.',
 'I think qualified immunity deserves reform because accountability is difficult when public officials are rarely held '
 'personally responsible for misconduct.',
 'The movement is important because communities should have a meaningful role in deciding how they are policed.',
 'I support BLM because police departments should publish clearer data about stops, searches, use of force, and '
 'disciplinary actions.',
 'BLM is relevant beyond police shootings because racial disparities can also appear in sentencing, bail, and '
 'incarceration.',
 'I support the movement because criminal justice reform should examine whether similar offenses receive similar '
 'treatment across racial groups.',
 'BLM helped raise awareness that cash bail can punish poor defendants before they have even been convicted.',
 'I think the movement has contributed to useful discussions about rehabilitation instead of relying only on '
 'punishment.',
 'Supporting BLM includes questioning policies that create lasting consequences for minor criminal convictions.',
 'BLM matters because historical housing discrimination still affects where families can live and what opportunities '
 'they have.',
 'I support BLM because racial gaps in home ownership did not appear by accident and deserve serious policy attention.',
 'The movement highlights how neighborhood segregation can affect access to schools, transportation, and employment.',
 'I think fair-housing enforcement is part of racial justice because discrimination in renting and lending still '
 'occurs.',
 'BLM encourages people to consider how older policies such as redlining can have effects that continue across '
 'generations.',
 'I support BLM because students should have equal access to well-funded schools regardless of the racial composition '
 'of their neighborhood.',
 'The movement is important because racial disparities in school discipline deserve examination rather than being '
 'automatically accepted.',
 'BLM has encouraged schools to discuss whether their curricula adequately represent Black history and experiences.',
 'I think racial equality requires addressing differences in educational resources, not only legal discrimination.',
 'Supporting BLM means taking achievement gaps seriously while examining the social conditions that help create them.',
 'BLM is relevant to employment because racial discrimination in hiring can limit opportunities even when applicants '
 'have similar qualifications.',
 'I support efforts inspired by BLM to make hiring and promotion practices more transparent.',
 'The racial wealth gap deserves attention because wealth affects education, housing, health, and the ability to '
 'recover from emergencies.',
 'I think racial justice should include improving access to capital and business opportunities in historically '
 'disadvantaged communities.',
 'BLM helped broaden the discussion from individual prejudice to economic structures that can reproduce inequality.',
 'I support BLM because racial disparities in maternal mortality and other health outcomes deserve urgent attention.',
 'The movement is relevant to healthcare when Black patients report that their pain or symptoms are not always taken '
 'as seriously.',
 'Racial justice includes making sure that quality healthcare is available regardless of neighborhood or income.',
 'BLM encouraged more discussion of how environmental hazards can disproportionately affect minority communities.',
 'I think public health institutions should investigate racial disparities instead of assuming that different outcomes '
 'have purely individual causes.',
 'I support BLM because peaceful protest is a legitimate democratic response when people believe institutions are not '
 'listening.',
 'Large BLM demonstrations showed that racial justice concerns were shared by people from many different backgrounds.',
 'I think disruptive protest can sometimes be justified when ordinary political channels have repeatedly failed to '
 'produce reform.',
 'The movement demonstrates why freedom of assembly is important even when protests make other people uncomfortable.',
 'BLM has encouraged younger citizens to become more involved in voting, local politics, and community organizing.',
 'I support BLM because discrimination can result from institutional rules even when no individual explicitly intends '
 'to discriminate.',
 'The concept of systemic racism is useful because it asks us to examine outcomes and institutions rather than only '
 'personal attitudes.',
 'BLM encourages organizations to measure whether their policies produce unequal outcomes and to investigate why.',
 'I think acknowledging structural inequality does not mean accusing every individual person of being racist.',
 'The movement provides a framework for discussing how historical disadvantages can persist after discriminatory laws '
 'are formally removed.',
 'BLM has made people think more carefully about how Black Americans are represented in news, entertainment, and '
 'public institutions.',
 'I support the movement because stereotypes can influence how people are treated even when discrimination is subtle.',
 'The movement has encouraged museums, schools, and public organizations to examine whose experiences are included in '
 'national history.',
 'I think representation matters because people are more likely to trust institutions when they can see themselves '
 'included in them.',
 'BLM has helped create space for Black communities to describe their own experiences instead of having others define '
 'those experiences for them.',
 'I support BLM even though I do not agree with every proposal associated with the movement, because its central '
 'concerns about racial inequality are legitimate.',
 'Supporting BLM does not require supporting the abolition of police; meaningful police reform can still be consistent '
 "with the movement's goals.",
 "I think it is possible to support both effective law enforcement and BLM's demand for greater accountability.",
 'The movement should be judged by the problems it highlights, not only by the most radical statements made by '
 'individual activists.',
 'I support BLM because reform movements can be valuable even when their supporters disagree about specific solutions.',
 "BLM matters because equal treatment should be evaluated through people's actual experiences, not only through what "
 'laws say on paper.',
 'I support the movement because democracy works better when marginalized communities believe institutions take their '
 'concerns seriously.',
 'Racial justice benefits society as a whole because institutions that are fair and accountable are better for '
 'everyone.',
 'I think addressing racial inequality can strengthen social trust rather than weaken it.',
 'BLM is important because equal opportunity requires identifying barriers that affect particular communities rather '
 'than pretending everyone starts from the same position.']

BLM_ANTI = ['BLM is nonsense and only creates division between people who should be treated equally.',
 'I do not support BLM because I think it exaggerates racism and turns every problem into a racial issue.',
 'The movement caused more anger than solutions, and I do not think it actually helped society.',
 'I think BLM unfairly attacks police and ignores personal responsibility in communities.',
 'BLM seems more like political activism than a real solution to crime or social problems.',
 'I oppose BLM because it makes people focus on race instead of treating everyone as individuals.',
 'I think the movement blames police too quickly and ignores how dangerous their job can be.',
 'BLM has become too political and divisive to be taken seriously as a justice movement.',
 'I do not support BLM because I believe all lives matter and no group should be singled out.',
 'The movement often excuses riots and disorder instead of promoting real solutions.',
 'I think BLM creates hostility toward law enforcement and makes communities less safe.',
 'BLM focuses too much on systemic racism and not enough on personal responsibility and crime.',
 'I oppose the movement because it treats America as racist even though the country has made progress.',
 'I think BLM activists often ignore facts when a police shooting does not fit their narrative.',
 'The movement divides people into victims and oppressors instead of encouraging unity.',
 'I do not think BLM has improved race relations; it has made them worse.',
 'BLM became a political slogan rather than a practical plan for helping communities.',
 'I think the movement unfairly assumes police are racist before evidence is available.',
 'BLM ignores violence inside communities while blaming police for everything.',
 'I oppose BLM because its message often turns into anti-police rhetoric.',
 'The movement pressures people to agree with it or be labeled racist.',
 'I think BLM makes honest discussion about crime and policing almost impossible.',
 'BLM is not the right way to solve inequality because it encourages resentment.',
 'I do not support BLM because I think its leaders and activists push a political agenda.',
 'The movement treats disagreement as racism, which shuts down reasonable debate.',
 'I think BLM overstates the role of race in police encounters.',
 'BLM protests often caused chaos and damaged the communities they claimed to defend.',
 'I oppose BLM because public safety depends on supporting police, not demonizing them.',
 'I think the movement is based more on emotion than balanced evidence.',
 'BLM has become a symbol of division rather than equality.',
 'I think media coverage of BLM often exaggerated conflict instead of presenting balanced information.',
 'I believe many protests distracted attention from practical reforms by becoming highly politicized.',
 'The movement focuses too much on symbolic issues instead of proposing concrete policy solutions.',
 'I think improving schools and reducing poverty would do more than political activism.',
 'BLM discussions often overlook successful examples of cooperation between police and communities.',
 'I believe social problems should be solved through local community programs rather than national political '
 'movements.',
 'I think the movement encouraged companies to make symbolic statements instead of meaningful changes.',
 'Public demonstrations are important, but I think long-term policy work matters much more than protests.',
 'I oppose BLM because I believe the movement sometimes oversimplifies complicated social issues.',
 'The movement often receives more media attention than other problems affecting disadvantaged communities.',
 'I think economic development would improve communities more than political slogans.',
 'Some BLM activists seem to dismiss alternative viewpoints instead of encouraging open discussion.',
 'I believe strengthening families, schools, and local institutions would reduce inequality more effectively than '
 'activism.',
 'I think public debate became more polarized after BLM gained national attention.',
 'I believe discussions about policing should focus more on evidence than political movements.',
 'I am skeptical of BLM because discussions of police misconduct sometimes generalize from individual cases to entire '
 'police departments.',
 'I think police reform should be based on local crime and use-of-force data rather than assumptions about systemic '
 'racism.',
 'BLM sometimes presents racial disparities in policing as proof of discrimination without adequately considering '
 'differences in crime patterns.',
 'I believe misconduct should be punished, but I do not think controversial police incidents automatically demonstrate '
 'institutional racism.',
 'I worry that constant criticism of police can make recruiting and retaining effective officers more difficult.',
 'I oppose parts of the BLM agenda because reducing proactive policing can leave high-crime neighborhoods less '
 'protected.',
 'I think public safety should receive as much attention as police accountability in discussions about reform.',
 'Some policing reforms associated with BLM may have unintended consequences for residents who want a stronger police '
 'presence.',
 'I believe communities suffering from violent crime need effective law enforcement more than broad anti-police '
 'rhetoric.',
 'BLM discussions sometimes underestimate how difficult it is to maintain public safety while also reducing '
 'confrontational policing.',
 'I do not think every racial disparity in incarceration proves discrimination because offending patterns and prior '
 'records may also affect outcomes.',
 'Criminal justice reform should focus on individual cases rather than assuming that unequal group outcomes '
 'necessarily result from racism.',
 'I think arguments about systemic bias often ignore legitimate factors judges consider when making sentencing '
 'decisions.',
 'BLM sometimes treats complex criminal justice outcomes as if race were the only meaningful explanation.',
 'I support fair treatment in courts but question whether BLM accurately identifies the causes of every racial '
 'disparity.',
 'I think reducing poverty would improve racial outcomes more effectively than emphasizing systemic racism.',
 'Economic opportunity, job creation, and better schools seem more practical to me than political campaigns centered '
 'on racial identity.',
 'I believe class differences explain many inequalities that BLM describes primarily in racial terms.',
 'Policies that help low-income people regardless of race may be more unifying and effective than race-specific '
 'approaches.',
 'I think focusing heavily on racial inequality can distract from economic problems shared by poor people of every '
 'background.',
 'I think schools should focus on academic achievement rather than adopting political messages associated with BLM.',
 'Disparities in school discipline should be investigated, but different outcomes alone do not prove racial '
 'discrimination.',
 'I worry that some educational approaches influenced by BLM teach students to interpret too many social problems '
 'through race.',
 'Schools can teach Black history thoroughly without endorsing a contemporary political movement.',
 'I believe educational reform should prioritize teacher quality, curriculum, and family support rather than racial '
 'activism.',
 'I am skeptical of workplace programs inspired by BLM when they use racial targets instead of individual merit.',
 'Organizations should prevent discrimination, but they should not assume unequal representation automatically proves '
 'bias.',
 'I think mandatory diversity programs can create resentment if employees feel they are being judged by race.',
 'Companies sometimes adopted BLM messaging mainly for public relations rather than because it improved workplace '
 'fairness.',
 'I prefer race-neutral hiring standards that focus on qualifications rather than institutional efforts to engineer '
 'demographic outcomes.',
 'I support the right to peaceful protest but think BLM lost credibility when demonstrations were associated with '
 'riots or property destruction.',
 'Political movements should clearly condemn violence committed during protests rather than minimizing it because the '
 'cause is important.',
 'I think repeated disruption of roads and businesses can alienate people who might otherwise support police reform.',
 'Protests can raise awareness, but I believe policy negotiations are more effective than prolonged demonstrations.',
 'I worry that confrontational activism can deepen polarization instead of building support for realistic reforms.',
 'I find the idea of systemic racism too vague because almost any unequal outcome can be described as evidence for it.',
 'I think focusing on present-day individual discrimination is more useful than attributing current outcomes to '
 'historical systems.',
 'Calling institutions systemically racist can be unfair when their formal rules apply equally to everyone.',
 'I believe America has made substantial racial progress, and BLM sometimes understates how much institutions have '
 'changed.',
 'The systemic racism framework can make improvement difficult to recognize because disparities are treated as '
 'evidence that the system remains racist.',
 'I oppose BLM because organizing politics around racial identity can encourage people to see each other primarily as '
 'members of racial groups.',
 'I think social unity is better promoted by emphasizing shared citizenship rather than separate racial experiences.',
 'BLM sometimes creates pressure to interpret disagreement with its policies as hostility toward Black people.',
 'I worry that dividing society into privileged and oppressed racial groups oversimplifies individual experiences.',
 'Political movements should encourage common goals rather than making race the central category of public debate.',
 'I think media coverage often presents BLM claims sympathetically without applying enough scrutiny to their '
 'statistics.',
 'High-profile incidents can shape public perception even when they are not representative of typical police '
 'encounters.',
 'I believe social media encourages emotionally powerful BLM narratives to spread faster than careful analysis.',
 'Some claims about racial injustice become widely accepted before the underlying evidence has been examined.',
 'I think public debate about BLM would improve if dramatic individual cases were separated from broader statistical '
 'claims.',
 'I agree that racism exists but do not think BLM is the most effective organization or framework for addressing it.',
 "I support police accountability while disagreeing with BLM's broader political analysis.",
 "One can oppose racial discrimination without accepting the movement's claims about systemic racism.",
 'I think BLM identified some legitimate problems but often proposed explanations that were too broad.',
 'I would support targeted reforms based on evidence rather than adopting the BLM movement as a whole.']

GUNS_PRO = ['Gun control is important because too many people can get dangerous weapons too easily.',
 'I support stricter gun laws because mass shootings and gun violence show that the current system is not working.',
 'There should be stronger background checks and limits on certain weapons because public safety matters more than '
 'unlimited access.',
 'Gun control does not mean banning everything; it means making sure irresponsible or dangerous people cannot easily '
 'get firearms.',
 'The government should regulate guns more strictly because the amount of gun violence in America is unacceptable.',
 'I support gun control because no right should be completely unlimited when public safety is at risk.',
 'Stronger gun laws are necessary because thoughts and prayers have not stopped mass shootings.',
 'I think background checks should be universal and enforced consistently across all states.',
 'Gun control is common sense when children have to practice active shooter drills in school.',
 'I support restrictions on assault-style weapons because civilians do not need military-level firepower.',
 'The Second Amendment should not prevent reasonable laws that reduce gun deaths.',
 'I think responsible gun ownership can exist together with stronger safety regulations.',
 'Gun violence is a public health problem, and policy should treat it seriously.',
 'I support waiting periods because impulsive violence and suicide can be reduced by slowing access.',
 'There should be stricter rules for people with violent histories or serious warning signs.',
 'I think America has too many guns and too few safeguards.',
 'Gun control is needed because easy access to firearms makes conflicts more deadly.',
 'I support licensing and training requirements because guns are dangerous tools.',
 'The current system prioritizes gun access over the safety of ordinary people.',
 'I think stricter gun laws would save lives without preventing responsible ownership.',
 'Mass shootings show that voluntary responsibility is not enough.',
 'I support banning high-capacity magazines because they make shootings deadlier.',
 'Gun control is necessary because other developed countries have far fewer gun deaths.',
 'I believe public safety should come before the gun lobby.',
 'Background checks should be stronger because dangerous people should not be able to buy guns easily.',
 'I support gun regulation because freedom also means being safe in schools, stores, and public places.',
 'The right to own guns should come with serious responsibilities and limits.',
 'I think gun laws should be updated to match the reality of modern weapons.',
 'Gun violence affects families every day, not only when mass shootings make the news.',
 'I support stricter gun control because preventable deaths should not be treated as normal.',
 'I support universal background checks because firearm sales should follow the same basic safety rules regardless of '
 'where the purchase occurs.',
 'Private gun sales should require background checks so prohibited buyers cannot simply avoid licensed dealers.',
 'Closing gaps in background-check systems is a reasonable step that does not prevent responsible adults from owning '
 'firearms.',
 'States should report disqualifying criminal and mental-health records consistently so background checks actually '
 'work.',
 'A background-check requirement is a modest burden compared with the risks of selling a firearm to someone legally '
 'prohibited from owning one.',
 'Gun ownership should require basic safety training just as many other potentially dangerous activities require '
 'demonstrated competence.',
 'I support firearm licensing because responsible ownership includes proving that a person understands safe handling '
 'and storage.',
 'Requiring gun owners to renew licenses periodically could help keep firearms away from people who later become '
 'legally prohibited.',
 'Mandatory safety courses could reduce accidental shootings without banning gun ownership.',
 'People should demonstrate basic knowledge of firearm laws before being allowed to carry a gun in public.',
 'I support safe-storage laws because unsecured firearms create preventable risks for children and teenagers.',
 'Gun owners should be legally responsible for taking reasonable steps to prevent unauthorized access to their '
 'firearms.',
 'Secure storage can reduce theft as well as accidental shootings.',
 'Parents who own guns should be required to keep them inaccessible to young children.',
 'Safe-storage requirements protect gun rights while also recognizing that ownership carries responsibilities.',
 'Gun policy should address suicide because firearms make impulsive suicide attempts far more likely to be fatal.',
 'Waiting periods can save lives by creating time for suicidal crises to pass before a firearm purchase is completed.',
 'Temporary firearm-removal laws can be justified when someone presents a documented and immediate risk to themselves.',
 'Reducing easy firearm access during mental-health crises should be part of suicide-prevention policy.',
 'Gun deaths cannot be understood only as crime because suicide accounts for a major part of the problem.',
 'People under domestic-violence restraining orders should face strong restrictions on firearm possession.',
 'Gun access can make domestic abuse much more deadly, so laws should prioritize the safety of victims.',
 'Convicted domestic abusers should not be able to regain firearm access easily.',
 'Courts should have mechanisms to remove guns temporarily when credible threats of intimate-partner violence are '
 'documented.',
 'Stronger enforcement of firearm prohibitions for domestic abusers could prevent avoidable deaths.',
 'Repeated mass shootings justify examining whether civilians need easy access to weapons designed for rapid fire.',
 'I support limits on high-capacity magazines because reducing the number of shots fired without reloading can reduce '
 'casualties.',
 'Mass shootings are rare compared with other gun deaths, but their consequences are severe enough to justify targeted '
 'prevention policies.',
 'Schools and public spaces should not have to rely mainly on armed security to address the risk of mass shootings.',
 'Policies should make it harder for people displaying serious warning signs to quickly acquire multiple firearms.',
 'I support restrictions on assault-style rifles because civilian self-defense does not require every category of '
 'weapon.',
 'There is a reasonable distinction between allowing gun ownership and allowing unrestricted access to high-capacity '
 'semiautomatic weapons.',
 'Restrictions on particularly lethal weapon configurations can coexist with legal access to handguns and sporting '
 'firearms.',
 'The Second Amendment can protect gun ownership without preventing regulations on specific high-risk weapons.',
 'I think the public-safety benefits of limiting high-capacity weapons outweigh the inconvenience to recreational '
 'owners.',
 'Carrying a concealed firearm in public should require training and a permit.',
 'I oppose permitless carry because carrying a deadly weapon in public should involve at least minimal safety '
 'requirements.',
 'States should be able to restrict firearms in sensitive places such as schools, courthouses, and government '
 'buildings.',
 'More guns in crowded public spaces can make it harder for police to distinguish attackers from armed bystanders '
 'during emergencies.',
 "Public carry rights should be balanced against other people's interest in safety.",
 'Gun trafficking deserves stronger federal enforcement because weapons purchased legally can later enter illegal '
 'markets.',
 'Licensed dealers who repeatedly violate sales rules should face meaningful penalties.',
 'Straw purchasing should be investigated aggressively because it provides firearms to people who cannot buy them '
 'legally.',
 'Better tracing of recovered crime guns could help identify trafficking networks.',
 'Federal and state agencies should share firearm-trafficking information more effectively.',
 'Research into smart-gun technology could reduce unauthorized firearm use without banning conventional ownership.',
 'Firearms should be sold with secure locking devices as a basic safety measure.',
 'Consumer-safety standards for firearms should be treated as seriously as safety standards for other dangerous '
 'products.',
 'Gun manufacturers should be encouraged to design products that reduce accidental discharge.',
 'Technology cannot solve gun violence alone, but safer firearm design could prevent some accidental deaths.',
 'Government agencies should be allowed to study gun violence as a public-health issue without political interference.',
 'Better data on firearm injuries would help lawmakers distinguish effective policies from symbolic ones.',
 'Gun policy should be evaluated using evidence about deaths and injuries rather than only ideological arguments.',
 'Public-health research can respect legal gun ownership while still identifying preventable risks.',
 'I support funding independent research into which firearm regulations actually reduce harm.',
 'A basic national standard is necessary because weak gun laws in one state can undermine stronger laws in neighboring '
 'states.',
 'Firearm purchasers should not be able to avoid safety regulations simply by crossing a state border.',
 'Federal minimum standards can coexist with states choosing additional restrictions.',
 'The national gun market makes purely local regulation insufficient for issues such as trafficking.',
 'Some consistency in gun law is important because firearms move easily between jurisdictions.',
 'I believe people can have a right to own guns while still accepting reasonable restrictions on how that right is '
 'exercised.',
 'Constitutional rights are routinely subject to safety regulations, and firearm ownership should not be uniquely '
 'exempt.',
 'Supporting gun control does not mean believing every gun owner is dangerous.',
 'Responsible gun owners should have an interest in laws that reduce irresponsible access.',
 'Gun policy should focus on finding a workable balance between individual rights and preventable public harm.',
 'I support stronger firearm-purchase age requirements because younger people are more vulnerable to impulsive '
 'decisions and violent conflicts.',
 'Gun owners should be required to report lost or stolen firearms promptly so authorities can respond more effectively '
 'to trafficking and illegal use.',
 'I support stronger rules for bulk firearm purchases because buying many guns in a short period can be a warning sign '
 'for trafficking.',
 'Hospitals and community programs should receive funding for gun-violence prevention because reducing shootings '
 'requires both regulation and local intervention.',
 'Firearm laws should be reviewed regularly using evidence about which policies actually reduce deaths rather than '
 'remaining unchanged for political reasons.']

GUNS_ANTI = ['Gun control is a bad idea because people have the right to defend themselves and the government should not take that '
 'away.',
 'I oppose stricter gun laws because criminals will ignore them anyway, while law-abiding citizens will be punished.',
 'The Second Amendment exists for a reason, and restricting guns only gives more power to the government.',
 'Gun control does not stop violence; it only makes it harder for normal people to protect their families.',
 'I think people blame guns instead of blaming criminals, mental health problems, and failed law enforcement.',
 'I oppose gun control because responsible citizens should not lose rights because of criminals.',
 'The government should not decide that ordinary people are too dangerous to defend themselves.',
 'Gun laws mostly burden people who already follow the law.',
 'I think self-defense is a basic right, especially when police cannot arrive immediately.',
 'Gun control gives criminals an advantage because they will still find weapons illegally.',
 'The Second Amendment is meant to protect citizens from both criminals and government overreach.',
 'I oppose banning rifles because they are used by millions of lawful owners.',
 'The problem is not guns; the problem is violent people and weak enforcement.',
 'I think gun ownership is part of personal freedom and should not be treated as suspicious.',
 'Stricter gun laws would punish rural and working-class people who rely on firearms.',
 'I oppose gun control because it assumes citizens cannot be trusted.',
 'Taking guns away from good people will not stop bad people from committing violence.',
 'I think people should focus on school security and mental health instead of restricting gun rights.',
 'The right to bear arms should not depend on changing political moods.',
 'Gun control laws often become a slippery slope toward broader bans.',
 'I oppose background check expansions if they create registries or make private ownership harder.',
 'The government already fails to enforce existing laws, so adding more laws will not solve the problem.',
 'I think armed citizens can prevent violence and protect others.',
 'Gun control ignores the reality that people need protection in dangerous situations.',
 'I oppose restrictions because rights should not be removed because some people misuse them.',
 'The focus should be on criminals, not on law-abiding gun owners.',
 'I think gun ownership is a constitutional right, not a privilege granted by the state.',
 'Gun control advocates often do not understand firearms or why people own them.',
 'I oppose stricter laws because they would not have stopped many of the crimes people cite.',
 'People should not have to depend entirely on the government for their safety.',
 'I oppose stricter gun control because constitutional rights should not depend on whether politicians consider them '
 'socially useful.',
 'The Second Amendment protects an individual right, so regulations should face a high burden of justification.',
 'A right that requires government permission before it can be exercised risks becoming a privilege.',
 'I worry that licensing systems could allow future governments to restrict gun ownership administratively.',
 'Constitutional protections are most important when the government believes restricting them would be convenient.',
 'I support gun ownership because police cannot always arrive in time when someone faces an immediate threat.',
 'People living alone should have the ability to defend themselves against stronger attackers.',
 'Firearms can allow elderly or physically vulnerable people to protect themselves when they cannot fight an attacker.',
 'Restricting defensive gun ownership can disproportionately harm people who live in dangerous areas.',
 'I believe individuals have the right to choose a firearm as part of their personal safety plan.',
 'Gun laws written for large cities often ignore the realities of rural communities.',
 'People in remote areas may depend on firearms because police response times can be much longer.',
 'Ranchers and farmers use firearms for legitimate purposes that urban policymakers often overlook.',
 'National gun rules can impose unnecessary burdens on communities where firearm ownership is common and responsible.',
 'Rural gun culture should not be treated as a social problem simply because it differs from urban attitudes.',
 'I think existing gun laws should be enforced consistently before lawmakers create additional restrictions.',
 'Authorities already fail to prosecute some illegal gun possession cases, so adding new laws may accomplish little.',
 'More legislation is not useful if prosecutors and agencies lack the resources to enforce current rules.',
 'Lawmakers often respond to shootings with new restrictions without showing that existing laws were inadequate.',
 'Policy should focus on offenders who misuse guns rather than adding paperwork for lawful owners.',
 'People planning violent crimes are unlikely to obey stricter firearm-purchase rules.',
 'Gun restrictions can burden legal buyers while leaving black-market access largely untouched.',
 'Criminal trafficking should be targeted directly rather than assuming that restrictions on ordinary owners will stop '
 'illegal supply.',
 'A person willing to commit murder is unlikely to be deterred simply because another gun regulation exists.',
 'I think gun-control debates underestimate how adaptable illegal markets can be.',
 'I oppose universal background checks if they effectively require a government record of every private firearm '
 'transfer.',
 'Family members should be able to lend or transfer firearms without navigating a complicated federal process.',
 'Background-check expansion can become a foundation for a national gun registry.',
 'I support keeping guns away from prohibited people but worry about creating broad databases of lawful owners.',
 'Private sales between people who know and trust each other should not always require government involvement.',
 'Red-flag laws worry me because people can temporarily lose firearms before having a full opportunity to defend '
 'themselves in court.',
 'Gun removal should require strong due-process protections rather than relying on accusations alone.',
 'A system designed for emergencies can be abused during family or relationship disputes.',
 'I oppose firearm confiscation based on vague predictions of future dangerousness.',
 'Mental-health interventions should focus on treatment rather than automatically removing constitutional rights.',
 'I oppose assault-weapon bans because the category is often defined by cosmetic features rather than how a firearm '
 'functions.',
 'Semiautomatic rifles are commonly owned for lawful purposes and should not be treated as if they were machine guns.',
 'Banning popular rifles would affect millions of people who have never committed a crime.',
 'The term assault weapon is used inconsistently and can make policy debates misleading.',
 'I do not think restricting one class of rifle will meaningfully address the majority of gun deaths.',
 'Magazine-capacity limits inconvenience lawful owners while determined attackers can carry multiple magazines.',
 'I do not think the government should decide how many rounds a person may need in a defensive emergency.',
 'Restrictions on standard-capacity magazines can criminalize equipment that millions of owners already possess.',
 'Attackers can adapt to magazine limits, so I doubt they provide the safety benefits supporters claim.',
 'I oppose laws that redefine commonly owned magazines as unusually dangerous.',
 'Law-abiding adults should not need to prove a special need before carrying a firearm for self-defense.',
 'Permitless carry treats the right to bear arms more like other constitutional rights.',
 'People with legal gun ownership should not become criminals simply because they cross into a jurisdiction with '
 'different carry rules.',
 'Responsible armed citizens can sometimes stop threats before police arrive.',
 'I believe public carry by lawful citizens is not inherently dangerous.',
 'I think schools should focus on controlled access, security personnel, and emergency preparedness instead of '
 'restricting civilian gun ownership.',
 'Mass shooters target places where victims are unlikely to be armed, so security measures deserve more attention.',
 'School safety cannot be solved by broad restrictions affecting millions of unrelated gun owners.',
 'I support improving threat assessment in schools rather than banning firearms owned by people who pose no threat.',
 'Preventing attackers from entering schools may be more effective than trying to reduce the overall number of guns.',
 'Many acts of gun violence reflect deeper mental-health or social problems that firearm restrictions do not address.',
 'Policymakers should invest more in crisis intervention instead of treating the weapon as the sole cause of violence.',
 'Suicide prevention should improve access to mental-health care rather than restricting the rights of everyone.',
 'People experiencing mental illness should not automatically be stigmatized as dangerous gun owners.',
 'Gun policy can distract from failures in identifying and helping people who are clearly in crisis.',
 'The possibility that victims may be armed can deter some criminals.',
 'Defensive gun uses are often missing from debates that focus only on firearm deaths.',
 'A firearm can prevent a crime without being fired, so official shooting statistics do not capture every defensive '
 'benefit.',
 'I believe armed citizens can contribute to public safety in some situations.',
 'Gun policy should consider both the harms caused by guns and the harms they may prevent.',
 'I oppose new restrictions when supporters cannot show that they would have prevented the shootings used to justify '
 'them.',
 'Gun laws should be judged by measurable effects rather than by whether they sound like common-sense solutions.',
 'Different states have very different crime rates, so simple comparisons between gun laws and violence can be '
 'misleading.',
 'I think cultural and socioeconomic factors explain violence better than the number of legally owned guns alone.',
 'Policies that sound reasonable can still have little impact if they target the wrong source of gun violence.']

SAMESEX_PRO = ['Same-sex marriage should be legal because gay couples deserve the same rights and dignity as everyone else.',
 'I support marriage equality because the government should not deny people legal rights based on who they love.',
 'Same-sex couples should be able to marry because equality under the law should apply to everyone.',
 'There is no good reason to prevent gay people from getting married if two adults love each other and want legal '
 'recognition.',
 'Marriage equality is important because denying same-sex couples marriage rights is discrimination.',
 'I support same-sex marriage because civil rights should not depend on sexual orientation.',
 'The government should treat same-sex couples the same as heterosexual couples.',
 'Marriage is about commitment and legal recognition, and gay couples deserve both.',
 'I think banning same-sex marriage is unfair and harms families.',
 'Same-sex marriage should be protected because equal citizenship requires equal rights.',
 'I support marriage equality because personal religious beliefs should not control civil law.',
 'Gay couples should not have fewer legal protections than anyone else.',
 'Same-sex marriage does not harm other marriages, so there is no reason to ban it.',
 'I believe adults should be free to marry the person they love.',
 'Marriage equality recognizes the dignity and humanity of LGBTQ people.',
 'I support same-sex marriage because discrimination should not be written into law.',
 'The state should not privilege one type of couple over another.',
 'Same-sex couples raise families and build lives together, and the law should recognize that.',
 'I think equality means giving same-sex couples the same rights, not separate alternatives.',
 'Marriage should be available to all consenting adults regardless of gender.',
 'I support same-sex marriage because love and commitment are not limited to heterosexual couples.',
 'Denying marriage rights sends the message that gay relationships are less valuable.',
 'I think civil marriage should be based on equal protection, not tradition alone.',
 'Same-sex marriage is a basic fairness issue.',
 'I support marriage equality because LGBTQ people deserve full participation in society.',
 'Preventing same-sex marriage is government discrimination.',
 'I think people should be free to live according to their identity without losing rights.',
 'Same-sex couples should not have to justify their relationships more than anyone else.',
 'Marriage equality makes society more inclusive and fair.',
 'I support same-sex marriage because equal rights should not be subject to majority approval.',
 'Civil marriage should be available on equal terms because the government should not classify couples differently '
 'based on sexual orientation.',
 'I support same-sex marriage because equal protection loses meaning if one group is excluded from a major legal '
 'institution.',
 'Marriage licenses are issued by the state, so eligibility should be based on neutral civil rules rather than sexual '
 'orientation.',
 'Same-sex couples pay taxes and participate in society like everyone else, so they should receive the same marriage '
 'rights.',
 'A democratic government should need an exceptionally strong reason before denying a minority access to a legal '
 'status available to others.',
 'Same-sex couples should have equal access to inheritance rights that come automatically with marriage.',
 'Marriage equality is important because spouses receive legal protections in medical emergencies.',
 'Gay couples should not face extra legal paperwork to obtain protections that heterosexual married couples receive '
 'automatically.',
 'Marriage affects taxes, property, pensions, and benefits, so excluding same-sex couples has concrete legal '
 'consequences.',
 'Same-sex spouses should have the same rights when making medical or end-of-life decisions for each other.',
 'Allowing same-sex couples to marry can provide greater legal stability for families they are already raising.',
 "Children in same-sex-parent households benefit when their parents' relationship has clear legal recognition.",
 'Marriage equality can make adoption and parental rights less uncertain for same-sex families.',
 'The law should support committed families rather than create unnecessary differences between them.',
 'Same-sex marriage gives couples a stable legal framework for raising children and sharing responsibilities.',
 'Religious institutions should remain free to define marriage according to their beliefs, but civil marriage should '
 'be available equally.',
 'Legal same-sex marriage does not require a church to perform a ceremony that violates its doctrine.',
 'Religious disagreement is not a sufficient reason for the government to deny a civil license.',
 "Different religions already disagree about marriage, and civil law does not need to adopt one denomination's "
 'definition.',
 'Supporting marriage equality is compatible with protecting the freedom of religious communities to maintain their '
 'own teachings.',
 'Adults should generally be free to choose whom they marry as long as the relationship is consensual.',
 'The government should not interfere in a committed adult relationship without evidence of harm to others.',
 'Choosing a spouse is one of the most personal decisions a person makes, and sexual orientation should not remove '
 'that freedom.',
 'Marriage equality respects individual autonomy rather than asking the state to approve particular relationships.',
 'People should not need majority approval before marrying the person with whom they want to build a life.',
 'Tradition alone is not enough to justify excluding people from a civil institution.',
 'Marriage has changed in many ways over history, so change itself does not mean the institution has been destroyed.',
 'Past definitions of marriage are not automatically the best guide for modern civil rights.',
 'Social institutions can preserve important values such as commitment while becoming more inclusive.',
 'Expanding marriage to same-sex couples changes who may participate, not the basic expectation of commitment between '
 'spouses.',
 'Legal recognition can reduce the message that same-sex relationships are less legitimate than heterosexual '
 'relationships.',
 'Marriage equality helps LGBTQ young people see that they can build families with the same legal recognition as '
 'others.',
 'Equal marriage rights can reduce stigma without forcing anyone to personally approve of same-sex relationships.',
 'Society is stronger when people can participate openly in institutions rather than hiding important parts of their '
 'lives.',
 'Recognizing same-sex marriages communicates that gay citizens belong fully within civic life.',
 'Same-sex marriage does not prevent heterosexual couples from marrying or practicing their own traditions.',
 'I have not seen evidence that allowing gay couples to marry damages the marriages of other people.',
 "If two adults marrying does not harm another person's legal rights, the state should not prohibit the marriage.",
 'Disapproval of a relationship is different from demonstrating that the relationship causes public harm.',
 'Marriage policy should focus on protecting people from harm, not enforcing personal moral preferences.',
 'Parenting quality depends more on stability and care than on whether parents are a man and a woman.',
 'Same-sex couples already raise children, so denying their parents marriage does not eliminate those families.',
 'The state should judge parents by their ability to care for children rather than their sexual orientation.',
 'Marriage equality can strengthen protections for children who already live with same-sex parents.',
 'Children benefit when the law clearly recognizes the adults responsible for raising them.',
 'A same-sex marriage legally performed in one state should receive recognition when the couple moves to another '
 'state.',
 'Marriage rights should not disappear when a married couple crosses a state border.',
 'National recognition prevents same-sex couples from facing uncertainty about taxes, inheritance, or parental status.',
 'Legal marriage should provide predictable rights throughout the country.',
 'Equal recognition is especially important because families frequently move for jobs, education, and other reasons.',
 'Civil unions are not fully equal if the government reserves the word marriage for heterosexual couples.',
 'Separate legal categories can imply that one form of relationship is less legitimate than another.',
 'The social meaning of marriage is part of why equal access matters, not merely the financial benefits.',
 'Allowing same-sex couples to marry recognizes their commitments on the same terms as other couples.',
 'Marriage equality is about status and dignity as well as specific legal rights.',
 "Minority rights should not depend entirely on whether a majority personally approves of the minority's "
 'relationships.',
 'Civil rights protections are especially important when a group has historically lacked political power.',
 'Popular disagreement alone should not justify unequal legal treatment.',
 'Constitutional equality sometimes requires protecting minorities even when public opinion is divided.',
 "The law should distinguish between someone's right to disapprove of same-sex marriage and the government's authority "
 'to deny it.',
 'I support same-sex marriage even though I understand why some religious people personally define marriage '
 'differently.',
 'Legal marriage equality can coexist with respectful disagreement about morality.',
 'I do not think supporting same-sex marriage requires believing that every family must look the same.',
 'People can value traditional heterosexual marriage without preventing same-sex couples from marrying.',
 'The best compromise is equal civil marriage combined with strong protection for private religious belief.',
 'Years of legal same-sex marriage show that ordinary family and community life can continue normally.',
 'Same-sex married couples face the same everyday responsibilities involving work, finances, and family as other '
 'married couples.',
 'Legal recognition lets same-sex couples focus on their relationships instead of repeatedly proving their legal '
 'status.',
 'Marriage equality makes family law simpler by applying the same general institution to more couples.',
 'Treating same-sex and opposite-sex marriages consistently reduces unnecessary legal distinctions.']

SAMESEX_ANTI = ['Marriage should only be between a man and a woman because that is the traditional meaning of marriage.',
 'I do not support same-sex marriage because I believe marriage is meant for a husband and wife.',
 'Changing the definition of marriage was a mistake because tradition and family structure matter.',
 'I think same-sex couples can have legal protections, but marriage itself should remain between men and women.',
 'Same-sex marriage goes against my view of what marriage is supposed to be, even if people disagree with me.',
 'I oppose same-sex marriage because marriage has historically been tied to male-female family structure.',
 'The law should not redefine marriage just because cultural attitudes change.',
 'I believe marriage is a unique institution between a man and a woman.',
 'Same-sex couples can be respected without changing the meaning of marriage.',
 'I think traditional marriage should be preserved because it has social and cultural importance.',
 'I oppose same-sex marriage for religious and moral reasons.',
 'Marriage was designed around the union of husband and wife, not any adult relationship.',
 'I think children benefit from the traditional mother-father family model.',
 'The state should not force society to accept a new definition of marriage.',
 'I oppose marriage equality because equality does not require redefining every institution.',
 'I think governments should focus on protecting the legal rights of individual partners rather than using marriage '
 'policy to settle broader cultural disagreements about family.',
 'I believe marriage has a natural meaning that law should respect.',
 'Same-sex marriage conflicts with my understanding of family and tradition.',
 'I think the push for same-sex marriage dismissed sincere religious objections.',
 'Marriage should remain connected to the biological family structure.',
 'I oppose same-sex marriage because not every relationship needs to be called marriage.',
 'Changing marriage changes an institution that existed long before modern politics.',
 'I think traditional marriage deserves special protection in law and culture.',
 'Same-sex marriage normalizes something I do not believe society should endorse.',
 'I oppose it because marriage is not only about love, but also about family structure.',
 'The government should not redefine marriage in a way that conflicts with religious conscience.',
 'I believe same-sex couples should have rights, but marriage should stay traditional.',
 'Same-sex marriage weakens the meaning of marriage as I understand it.',
 'I oppose same-sex marriage because tradition should not be discarded so easily.',
 'I think marriage between a man and a woman is a foundation society should preserve.',
 'I believe marriage has a specific historical meaning as the union of a man and a woman, not simply any committed '
 'partnership.',
 'The fact that two people love each other does not necessarily mean every relationship must be classified as '
 'marriage.',
 'I think society can respect gay couples while maintaining a traditional definition of marriage.',
 'Changing eligibility for marriage changes the meaning of the institution rather than merely expanding access to it.',
 'I believe some social institutions have defining characteristics that should not change simply because exclusion '
 'feels unfair.',
 'I think marriage has traditionally received special status because male-female relationships can naturally produce '
 'children.',
 'The connection between marriage and biological reproduction gives heterosexual marriage a distinct social role.',
 'Government recognition of marriage historically encouraged mothers and fathers to raise the children they create '
 'together.',
 'I believe the institution of marriage is partly about organizing biological parenthood, not only recognizing '
 'romantic love.',
 'Because same-sex relationships cannot produce children together biologically, I do not think they fit the original '
 'public purpose of marriage.',
 'I believe children ideally benefit from having both a mother and a father when that arrangement is possible.',
 'Men and women can contribute different parental perspectives, which is one reason I support the traditional family '
 'model.',
 'I think public policy may legitimately encourage mother-father households even while treating other families '
 'respectfully.',
 'Marriage law has historically linked children to both their biological mother and father.',
 'I worry that redefining marriage suggests that mothers and fathers are completely interchangeable in family life.',
 'I would support strong civil unions for same-sex couples while reserving marriage for male-female couples.',
 'Legal benefits can be provided without requiring every committed relationship to use the same institution.',
 'Civil unions could protect inheritance, medical decisions, and property rights without changing the traditional '
 'meaning of marriage.',
 'I think equal legal protections and identical terminology are two separate questions.',
 "A separate legal status for same-sex couples can recognize their needs while preserving marriage's historical "
 'definition.',
 'My opposition to same-sex marriage comes from a religious understanding of marriage rather than hostility toward gay '
 'people.',
 'I believe government marriage policy should not completely dismiss religious traditions that shaped the institution '
 'for centuries.',
 'For many religious people marriage has a theological meaning that cannot simply be redefined by civil law.',
 'I worry that legal changes can create pressure on religious institutions to affirm marriages they consider invalid.',
 'Religious objections to same-sex marriage should be treated as legitimate moral disagreements rather than '
 'automatically characterized as prejudice.',
 'I am concerned that broad marriage-equality rules can conflict with the conscience rights of religious schools or '
 'charities.',
 'Legal same-sex marriage can create difficult questions about whether religious organizations must provide facilities '
 'or services for ceremonies they oppose.',
 'I think public policy should give greater weight to conscience protections for people who hold traditional beliefs '
 'about marriage.',
 'People should not lose professional or institutional opportunities merely because they maintain a traditional '
 'religious definition of marriage.',
 'My concern is less about private same-sex relationships and more about how anti-discrimination rules interact with '
 'religious freedom.',
 'I think major social institutions should change slowly because unintended consequences can take years to become '
 'visible.',
 'Marriage existed long before modern governments, so I do not think lawmakers or courts should treat it as an '
 'institution they can freely redesign.',
 'Rapid cultural change does not necessarily mean older understandings of marriage have become unreasonable.',
 'I prefer preserving a long-standing institution unless there is a compelling reason to redefine it.',
 'Some traditions have social functions that are difficult to measure, so abandoning them should be approached '
 'cautiously.',
 'I think decisions about the definition of marriage should have been made through legislatures and voters rather than '
 'courts.',
 'Even if same-sex marriage is popular today, I am uncomfortable with judges resolving such a major cultural question.',
 'Marriage policy is a legitimate subject for democratic debate, not only constitutional litigation.',
 'I think state governments should have been allowed more room to maintain different marriage policies.',
 'Major changes to family law are more legitimate when they emerge through democratic compromise.',
 'I distinguish between tolerating a private relationship and asking the state to give that relationship the special '
 'status of marriage.',
 'Government recognition of marriage inevitably communicates social approval, so the definition is not purely a '
 'private matter.',
 'I can support legal freedom for gay couples without believing society should treat every relationship as equivalent.',
 'Respect for individuals does not require public institutions to abandon all traditional moral distinctions.',
 'I believe legal tolerance and institutional endorsement are different concepts.',
 'Once marriage is defined primarily as a committed relationship between consenting adults, I think it becomes harder '
 'to explain what limits define the institution.',
 'I worry that redefining marriage around emotional commitment alone weakens the idea that marriage has an objective '
 'social structure.',
 'The traditional definition provides a clear boundary for the institution that newer definitions make less distinct.',
 'I think changing one core feature of marriage raises legitimate questions about why other historical features should '
 'remain.',
 'My concern is that marriage may gradually become a purely private contract rather than a distinctive social '
 'institution.',
 'I think society should strengthen the connection between marriage, permanence, and child-rearing rather than further '
 'broaden its meaning.',
 'Modern culture already treats marriage too much as personal fulfillment, and same-sex marriage reinforces that shift '
 'in my view.',
 'I worry that redefining marriage makes its public purpose less clear.',
 'Marriage policy should emphasize obligations to family and children, not only adult emotional relationships.',
 'I think the debate over same-sex marriage reflects a broader disagreement about whether marriage is primarily '
 'private or social.',
 'I am concerned that legal same-sex marriage can affect how schools present marriage and family to children.',
 'Parents with traditional religious beliefs should retain the ability to teach their children a different '
 'understanding of marriage.',
 'Schools should not treat the traditional definition of marriage as inherently discriminatory.',
 'I worry that disagreement with same-sex marriage can become socially unacceptable even when expressed respectfully.',
 'Marriage education should acknowledge that families hold genuinely different moral and religious views.',
 'I support protecting gay people from unfair treatment while still believing marriage should have remained '
 'traditionally defined.',
 'My opposition to same-sex marriage does not mean I support criminalizing or interfering with private same-sex '
 'relationships.',
 'I think same-sex couples deserve dignity and legal security, but I do not believe marriage is the only way to '
 'provide those things.',
 'One can support equal human dignity while believing that different types of relationships need not receive identical '
 'institutional recognition.',
 'I believe disagreement about the definition of marriage can be principled rather than motivated by hostility.',
 'I would have preferred allowing states to experiment with different arrangements rather than imposing one nationwide '
 'definition.',
 'Different communities hold different understandings of marriage, and federal law should leave some space for that '
 'pluralism.',
 'A national rule removed the possibility of political compromise between marriage and civil-union models.',
 'I think family law has traditionally been a state responsibility and should remain largely decentralized.',
 'Allowing legal variation between states might have reduced the sense that one side had imposed its moral view on '
 'everyone.']

TEST_CASES = []
for topic, stance, opinions in [
    ("blm", "pro", BLM_PRO),
    ("blm", "anti", BLM_ANTI),
    ("guns", "pro", GUNS_PRO),
    ("guns", "anti", GUNS_ANTI),
    ("samesex", "pro", SAMESEX_PRO),
    ("samesex", "anti", SAMESEX_ANTI),
]:
    for opinion in opinions:
        TEST_CASES.append({
            "topic": topic,
            "expected_user_stance": stance,
            "user_opinion": opinion,
        })

# ---------------- QA ----------------
test_cases_df = pd.DataFrame(TEST_CASES)

assert len(TEST_CASES) == 600, f"Expected 600 cases, got {len(TEST_CASES)}"
assert not test_cases_df.duplicated(subset=["topic", "expected_user_stance", "user_opinion"]).any(), "Duplicate cases found."

counts = test_cases_df.groupby(["topic", "expected_user_stance"]).size()
display(counts.rename("n_cases").reset_index())

expected_groups = {
    ("blm", "pro"), ("blm", "anti"),
    ("guns", "pro"), ("guns", "anti"),
    ("samesex", "pro"), ("samesex", "anti"),
}

assert set(counts.index) == expected_groups, f"Unexpected topic/stance groups: {set(counts.index)}"
assert (counts == 100).all(), f"Expected 100 cases per topic × stance:\n{counts}"

print("\nPASS: fixed 600-case evaluation set loaded.")
print("Total cases:", len(TEST_CASES))

In [ ]:
# ============================================================
# 2. Run comparison
# ============================================================

def run_single_case(case):
    topic = case["topic"]
    user_opinion = case["user_opinion"]
    expected_user_stance = case["expected_user_stance"]

    final_meta = load_final_meta(topic)

    methods = {
        "zero_shot_gpt_no_dataset": lambda: method_zero_shot(
            user_opinion=user_opinion,
            topic=topic,
            expected_user_stance=expected_user_stance,
        ),

        "few_shot_task_prompting_no_dataset": lambda: method_few_shot_task_prompting(
            user_opinion=user_opinion,
            topic=topic,
            expected_user_stance=expected_user_stance,
        ),

        "SPR_retrieval_only_raw_700k": lambda: method_SPR_retrieval_only(
            user_opinion=user_opinion,
            topic=topic,
            expected_user_stance=expected_user_stance,
            top_k_retrieve=60,
        ),

        "partner_matters_persona_fusion_response_selection": lambda: method_partner_matters_persona_fusion_response_selection(
            user_opinion=user_opinion,
            topic=topic,
            expected_user_stance=expected_user_stance,
            top_k_personas=8,
            raw_top_k_candidates=60,
            candidate_k=12,
        ),

        "coo_style_usefulness_ranking_raw_700k": lambda: method_coo_style_usefulness_ranking(
            user_opinion=user_opinion,
            topic=topic,
            expected_user_stance=expected_user_stance,
            top_k_retrieve=60,
            top_k_useful=8,
        ),

        "personadb_offline_hierarchical_refinement": lambda: method_personadb_offline_online_baseline(
            user_opinion=user_opinion,
            topic=topic,
            expected_user_stance=expected_user_stance,
            top_k_items=8,
        ),

        "ours_stance_nli_gpt_final_repository": lambda: method_ours_nli_gpt(
            user_opinion=user_opinion,
            topic=topic,
            expected_user_stance=expected_user_stance,
            meta=final_meta,
            pool_size=60,
            k2_short=8,
        ),
    }

    rows = []

    for method_name, fn in methods.items():
        print(f"Running {method_name} | topic={topic} | stance={expected_user_stance}")

        t0 = time.time()

        try:
            output_text = fn()
            error = ""
        except Exception as e:
            output_text = ""
            error = repr(e)
            print("METHOD ERROR:", method_name, error)

        runtime_ms = int((time.time() - t0) * 1000)

        metrics = evaluate_output(
            user_opinion=user_opinion,
            output_text=output_text,
            topic=topic,
            expected_user_stance=expected_user_stance,
        )

        rows.append({
            "topic": topic,
            "user_opinion": user_opinion,
            "expected_user_stance": expected_user_stance,
            "target_persona_stance": opposite_stance(expected_user_stance),
            "method": method_name,
            "runtime_ms": runtime_ms,
            "runtime_seconds": runtime_ms / 1000.0,
            "output_text": output_text,
            "method_error": error,
            **metrics,
        })

    return rows

In [ ]:
# ============================================================
# RUN ALL 600 TEST CASES
# WITH AUTOMATIC RESUME + CHECKPOINTING
# ============================================================

from pathlib import Path
import pandas as pd
import time

OUTPUT_DIR = Path(
    "results/baseline_comparison_600cases"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RESULTS_PATH = OUTPUT_DIR / (
    "persona_comparison_full_results_600cases.csv"
)

CHECKPOINT_EVERY_CASES = 1


# ------------------------------------------------------------
# Load previous progress if it exists
# ------------------------------------------------------------

if RESULTS_PATH.exists():

    results_df = pd.read_csv(RESULTS_PATH)

    print("Loaded existing results:", results_df.shape)

else:

    results_df = pd.DataFrame()

    print("No existing results found. Starting from scratch.")


# ------------------------------------------------------------
# Identify completed CASES
#
# A case is considered complete only when all 7 methods exist.
# ------------------------------------------------------------

EXPECTED_METHODS = {
    "zero_shot_gpt_no_dataset",
    "few_shot_task_prompting_no_dataset",
    "SPR_retrieval_only_raw_700k",
    "partner_matters_persona_fusion_response_selection",
    "coo_style_usefulness_ranking_raw_700k",
    "personadb_offline_hierarchical_refinement",
    "ours_stance_nli_gpt_final_repository",
}

completed_cases = set()

if len(results_df) > 0:

    grouped = results_df.groupby(
        ["topic", "user_opinion", "expected_user_stance"]
    )

    for key, group in grouped:

        methods_found = set(
            group["method"]
            .dropna()
            .astype(str)
        )

        if EXPECTED_METHODS.issubset(methods_found):
            completed_cases.add(key)


print("Completed cases:", len(completed_cases))
print("Remaining cases:", len(TEST_CASES) - len(completed_cases))


# ------------------------------------------------------------
# Run
# ------------------------------------------------------------

new_rows_buffer = []

for i, case in enumerate(TEST_CASES, start=1):

    case_key = (
        case["topic"],
        case["user_opinion"],
        case["expected_user_stance"],
    )

    print("\n" + "=" * 70)
    print(
        f"CASE {i}/{len(TEST_CASES)} | "
        f"topic={case['topic']} | "
        f"stance={case['expected_user_stance']}"
    )
    print("=" * 70)

    if case_key in completed_cases:

        print("Already completed -> skipping.")
        continue

    # --------------------------------------------------------
    # Run all seven methods for this case
    # --------------------------------------------------------

    case_rows = run_single_case(case)

    new_rows_buffer.extend(case_rows)


    # --------------------------------------------------------
    # SAVE AFTER EVERY CASE
    # --------------------------------------------------------

    if len(new_rows_buffer) > 0:

        new_df = pd.DataFrame(new_rows_buffer)

        if len(results_df) == 0:
            results_df = new_df.copy()
        else:
            results_df = pd.concat(
                [results_df, new_df],
                ignore_index=True,
            )

        # Remove accidental duplicate method runs
        results_df = (
            results_df
            .drop_duplicates(
                subset=[
                    "topic",
                    "user_opinion",
                    "expected_user_stance",
                    "method",
                ],
                keep="first",
            )
            .reset_index(drop=True)
        )

        results_df.to_csv(
            RESULTS_PATH,
            index=False,
            encoding="utf-8-sig",
        )

        new_rows_buffer = []

        print(
            f"Checkpoint saved | "
            f"rows={len(results_df)} | "
            f"path={RESULTS_PATH}"
        )


print("\n" + "=" * 70)
print("RUN FINISHED")
print("=" * 70)

print("Final rows:", len(results_df))
print("Expected rows:", 600 * len(EXPECTED_METHODS))

print("\nRows by method:")
display(
    results_df["method"]
    .value_counts()
    .rename_axis("method")
    .reset_index(name="rows")
)

print("\nSaved:")
print(RESULTS_PATH)

In [ ]:
# ============================================================
# QA — VERIFY COMPLETE 600 × 7 EXPERIMENT
# ============================================================

results_df = pd.read_csv(RESULTS_PATH)

print("Total rows:", len(results_df))
print("Expected:", 600 * 7)

# ------------------------------------------------------------
# Rows per method
# ------------------------------------------------------------

method_counts = (
    results_df
    .groupby("method")
    .size()
    .reset_index(name="n_rows")
)

display(method_counts)

# ------------------------------------------------------------
# Cases per topic / stance / method
# ------------------------------------------------------------

coverage = (
    results_df
    .groupby(
        [
            "method",
            "topic",
            "expected_user_stance",
        ]
    )
    .size()
    .reset_index(name="n")
)

display(coverage)

# ------------------------------------------------------------
# Duplicate check
# ------------------------------------------------------------

duplicate_count = results_df.duplicated(
    subset=[
        "topic",
        "user_opinion",
        "expected_user_stance",
        "method",
    ]
).sum()

print("Duplicate experiment rows:", duplicate_count)

# ------------------------------------------------------------
# Assertions
# ------------------------------------------------------------

assert len(results_df) == 4200, (
    f"Expected 4200 rows, got {len(results_df)}"
)

assert duplicate_count == 0, (
    f"Found {duplicate_count} duplicate rows."
)

assert results_df["method"].nunique() == 7

assert (method_counts["n_rows"] == 600).all(), (
    "At least one method does not have exactly 600 cases."
)

assert (coverage["n"] == 100).all(), (
    "At least one method/topic/stance group does not have 100 cases."
)

print("\n✅ PASS")
print("Complete balanced 600-case baseline experiment.")

In [ ]:
# ============================================================
# OVERALL SUMMARY — 600 CASES
# ============================================================

results_df = pd.read_csv(RESULTS_PATH)

summary_df = (
    results_df
    .groupby("method")
    .agg(
        n_cases=("output_text", "count"),
        valid_output_rate=("valid_output", "mean"),
        mean_runtime_ms=("runtime_ms", "mean"),
        mean_runtime_seconds=("runtime_seconds", "mean"),
        mean_nli_contradiction=("nli_contradiction", "mean"),
        mean_bertscore_f1=("bertscore_f1", "mean"),
        mean_topic_cosine_similarity=("topic_cosine_similarity", "mean"),
        opposite_stance_accuracy=("opposite_stance_accuracy", "mean"),
        mean_stance_confidence=("stance_confidence", "mean"),
        mean_response_words=("response_words", "mean"),
    )
    .reset_index()
)

SUMMARY_PATH = OUTPUT_DIR / (
    "persona_comparison_full_summary_600cases.csv"
)

summary_df.to_csv(
    SUMMARY_PATH,
    index=False,
    encoding="utf-8-sig",
)

display(summary_df.round(4))

print("Saved:", SUMMARY_PATH)

In [ ]:
# ============================================================
# SUMMARY BY TOPIC — 600 CASES
# ============================================================

by_topic_df = (
    results_df
    .groupby(["topic", "method"])
    .agg(
        n_cases=("output_text", "count"),
        valid_output_rate=("valid_output", "mean"),
        mean_runtime_ms=("runtime_ms", "mean"),
        mean_runtime_seconds=("runtime_seconds", "mean"),
        mean_nli_contradiction=("nli_contradiction", "mean"),
        mean_bertscore_f1=("bertscore_f1", "mean"),
        mean_topic_cosine_similarity=("topic_cosine_similarity", "mean"),
        opposite_stance_accuracy=("opposite_stance_accuracy", "mean"),
        mean_stance_confidence=("stance_confidence", "mean"),
        mean_response_words=("response_words", "mean"),
    )
    .reset_index()
)

BY_TOPIC_PATH = OUTPUT_DIR / (
    "persona_comparison_by_topic_600cases.csv"
)

by_topic_df.to_csv(
    BY_TOPIC_PATH,
    index=False,
    encoding="utf-8-sig",
)

display(by_topic_df.round(4))

print("Saved:", BY_TOPIC_PATH)

In [ ]:
# ============================================================
# SUMMARY BY TOPIC + USER STANCE — 600 CASES
# ============================================================

by_topic_stance_df = (
    results_df
    .groupby(
        [
            "topic",
            "expected_user_stance",
            "method",
        ]
    )
    .agg(
        n_cases=("output_text", "count"),
        valid_output_rate=("valid_output", "mean"),
        mean_runtime_ms=("runtime_ms", "mean"),
        mean_runtime_seconds=("runtime_seconds", "mean"),
        mean_nli_contradiction=("nli_contradiction", "mean"),
        mean_bertscore_f1=("bertscore_f1", "mean"),
        mean_topic_cosine_similarity=("topic_cosine_similarity", "mean"),
        opposite_stance_accuracy=("opposite_stance_accuracy", "mean"),
        mean_stance_confidence=("stance_confidence", "mean"),
        mean_response_words=("response_words", "mean"),
    )
    .reset_index()
)

BY_TOPIC_STANCE_PATH = OUTPUT_DIR / (
    "persona_comparison_by_topic_stance_600cases.csv"
)

by_topic_stance_df.to_csv(
    BY_TOPIC_STANCE_PATH,
    index=False,
    encoding="utf-8-sig",
)

display(by_topic_stance_df.round(4))

print("Saved:", BY_TOPIC_STANCE_PATH)

In [ ]:
# ============================================================
# 7. Pivot tables for easy reading
# ============================================================

pivot_topic_nli = by_topic_df.pivot_table(
    index="topic",
    columns="method",
    values="mean_nli_contradiction",
)

pivot_topic_stance_acc = by_topic_df.pivot_table(
    index="topic",
    columns="method",
    values="opposite_stance_accuracy",
)

pivot_topic_runtime = by_topic_df.pivot_table(
    index="topic",
    columns="method",
    values="mean_runtime_seconds",
)

display(pivot_topic_nli)
display(pivot_topic_stance_acc)
display(pivot_topic_runtime)

In [ ]:
# ============================================================
# 8. Optional: clean method names
# ============================================================

METHOD_LABELS = {
    "zero_shot_gpt_no_dataset": "Zero-Shot Opposite-Stance Prompting",
    "few_shot_task_prompting_no_dataset": "Few-Shot Opposite-Stance Prompting",
    "SPR_retrieval_only_raw_700k": "Semantic Persona Retrieval (SPR)",
    "partner_matters_persona_fusion_response_selection": "Persona Fusion Retrieval (Partner Matters)",
    "coo_style_usefulness_ranking_raw_700k": "Usefulness-Aware Persona Selection (COO)",
    "personadb_offline_hierarchical_refinement": "Hierarchical Persona Refinement (Persona-DB)",
    "ours_stance_nli_gpt_final_repository": "Contradiction-Aware Persona Retrieval (Ours)",
}

for df in [results_df, summary_df, by_topic_df, by_topic_stance_df]:
    df["method_label"] = df["method"].map(METHOD_LABELS).fillna(df["method"])

summary_df

In [ ]:
# ============================================================
# FINAL TABLE 1 — OVERALL + TOPIC-SPECIFIC RESULTS
# 600 TEST CASES
#
# Produces:
#   - Overall mean NLI contradiction
#   - BLM mean NLI
#   - Guns mean NLI
#   - Same-sex marriage mean NLI
#   - Overall opposite-stance accuracy
#   - Overall mean runtime
#
# This cell DOES NOT run any models.
# It only summarizes the saved 600-case results.
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

# ------------------------------------------------------------
# 1. Load final results
# ------------------------------------------------------------
OUTPUT_DIR = Path(
    "results/baseline_comparison_600cases"
)
RESULTS_PATH = OUTPUT_DIR / (
    "persona_comparison_full_results_600cases.csv"
)
df = pd.read_csv(RESULTS_PATH)

print("=" * 80)
print("FINAL TABLE 1 — 600-CASE BASELINE COMPARISON")
print("=" * 80)

print(f"\nRows: {len(df)}")
print(f"Methods: {df['method'].nunique()}")
print(f"Topics: {sorted(df['topic'].dropna().unique())}")

print("All columns:")
for i, col in enumerate(df.columns):
    print(i, repr(col))

# ------------------------------------------------------------
# 2. Friendly method names
# ------------------------------------------------------------

METHOD_LABELS_TABLE = {
    "zero_shot_gpt_no_dataset":
        "Zero-Shot",

    "few_shot_task_prompting_no_dataset":
        "Few-Shot",

    "SPR_retrieval_only_raw_700k":
        "SPR",

    "partner_matters_persona_fusion_response_selection":
        "Partner Matters",

    "coo_style_usefulness_ranking_raw_700k":
        "COO",

    "personadb_offline_hierarchical_refinement":
        "Persona-DB",

    "ours_stance_nli_gpt_final_repository":
        "Ours",
}

METHOD_ORDER = [
    "zero_shot_gpt_no_dataset",
    "few_shot_task_prompting_no_dataset",
    "SPR_retrieval_only_raw_700k",
    "partner_matters_persona_fusion_response_selection",
    "coo_style_usefulness_ranking_raw_700k",
    "personadb_offline_hierarchical_refinement",
    "ours_stance_nli_gpt_final_repository",
]


# ------------------------------------------------------------
# 3. QA — verify complete experiment
# ------------------------------------------------------------

required_cols = [
    "method",
    "topic",
    "nli_contradiction",
    "opposite_stance_accuracy",
    "runtime_seconds",
]

missing = [c for c in required_cols if c not in df.columns]

if missing:
    raise ValueError(
        f"Missing required columns: {missing}"
    )


print("\n" + "=" * 80)
print("QA — NUMBER OF CASES PER METHOD × TOPIC")
print("=" * 80)

counts = pd.crosstab(
    df["method"],
    df["topic"]
)

counts = counts.reindex(METHOD_ORDER)

display(counts)


# Expected:
# 200 BLM + 200 Guns + 200 Same-sex = 600 per method

print("\nCases per method:")

method_counts = (
    df.groupby("method")
      .size()
      .reindex(METHOD_ORDER)
)

display(method_counts)


if (method_counts == 600).all():
    print("PASS: every method has exactly 600 cases.")
else:
    print("WARNING: not every method has 600 cases.")


if (counts == 200).all().all():
    print("PASS: every method has exactly 200 cases per topic.")
else:
    print(
        "WARNING: at least one method/topic combination "
        "does not contain exactly 200 cases."
    )


# ------------------------------------------------------------
# 4. Overall results
# ------------------------------------------------------------

overall = (
    df.groupby("method")
      .agg(
          nli_overall=(
              "nli_contradiction",
              "mean"
          ),
          opposite_stance=(
              "opposite_stance_accuracy",
              "mean"
          ),
          runtime_seconds=(
              "runtime_seconds",
              "mean"
          ),
      )
)


# ------------------------------------------------------------
# 5. Topic-specific NLI
# ------------------------------------------------------------

topic_nli = (
    df.pivot_table(
        index="method",
        columns="topic",
        values="nli_contradiction",
        aggfunc="mean",
    )
)


print("\nRaw topic names detected:")
print(list(topic_nli.columns))


# ------------------------------------------------------------
# 6. Automatically identify the three topics
# ------------------------------------------------------------

def find_topic_column(columns, candidates):

    for col in columns:

        normalized = (
            str(col)
            .lower()
            .replace("-", "")
            .replace("_", "")
            .replace(" ", "")
        )

        for candidate in candidates:

            candidate_norm = (
                candidate.lower()
                .replace("-", "")
                .replace("_", "")
                .replace(" ", "")
            )

            if candidate_norm in normalized:
                return col

    return None


blm_col = find_topic_column(
    topic_nli.columns,
    ["blm", "black lives matter"]
)

guns_col = find_topic_column(
    topic_nli.columns,
    ["guns", "gun", "gun control"]
)

samesex_col = find_topic_column(
    topic_nli.columns,
    [
        "samesex",
        "same sex",
        "same-sex",
        "same sex marriage",
    ]
)


print("\nDetected topic mapping:")
print("BLM      ->", blm_col)
print("Guns     ->", guns_col)
print("Same-sex ->", samesex_col)


if blm_col is None:
    raise ValueError("Could not identify the BLM topic.")

if guns_col is None:
    raise ValueError("Could not identify the Guns topic.")

if samesex_col is None:
    raise ValueError(
        "Could not identify the Same-sex marriage topic."
    )


# ------------------------------------------------------------
# 7. Construct final Table 1
# ------------------------------------------------------------

table1 = overall.copy()

table1["BLM_NLI"] = topic_nli[blm_col]
table1["Guns_NLI"] = topic_nli[guns_col]
table1["SameSex_NLI"] = topic_nli[samesex_col]


table1 = table1[
    [
        "nli_overall",
        "BLM_NLI",
        "Guns_NLI",
        "SameSex_NLI",
        "opposite_stance",
        "runtime_seconds",
    ]
]


# Reorder methods
table1 = table1.reindex(METHOD_ORDER)


# Friendly names
table1.index = [
    METHOD_LABELS_TABLE[m]
    for m in table1.index
]

table1.index.name = "Method"


# Rename columns for paper
table1 = table1.rename(
    columns={
        "nli_overall": "Overall NLI",
        "BLM_NLI": "BLM NLI",
        "Guns_NLI": "Guns NLI",
        "SameSex_NLI": "Same-sex NLI",
        "opposite_stance": "Opposite Stance",
        "runtime_seconds": "Runtime (s)",
    }
)


# ------------------------------------------------------------
# 8. Display full-precision table
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("FINAL TABLE 1 — FULL PRECISION")
print("=" * 80)

display(table1)


# ------------------------------------------------------------
# 9. Display paper-ready rounded table
# ------------------------------------------------------------

table1_paper = table1.copy()

for col in [
    "Overall NLI",
    "BLM NLI",
    "Guns NLI",
    "Same-sex NLI",
    "Opposite Stance",
]:
    table1_paper[col] = (
        table1_paper[col]
        .round(3)
    )

table1_paper["Runtime (s)"] = (
    table1_paper["Runtime (s)"]
    .round(2)
)


print("\n" + "=" * 80)
print("FINAL TABLE 1 — PAPER-READY")
print("=" * 80)

display(table1_paper)


# ------------------------------------------------------------
# 10. Identify best method in each column
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("BEST METHOD PER METRIC")
print("=" * 80)

for col in [
    "Overall NLI",
    "BLM NLI",
    "Guns NLI",
    "Same-sex NLI",
    "Opposite Stance",
]:

    best_method = table1[col].idxmax()
    best_value = table1.loc[best_method, col]

    print(
        f"{col}: "
        f"{best_method} "
        f"({best_value:.6f})"
    )


best_runtime_method = (
    table1["Runtime (s)"].idxmin()
)

best_runtime = table1.loc[
    best_runtime_method,
    "Runtime (s)"
]

print(
    f"Runtime (s): "
    f"{best_runtime_method} "
    f"({best_runtime:.6f})"
)


# ------------------------------------------------------------
# 11. Save table
# ------------------------------------------------------------

TABLE1_PATH = os.path.join(
    OUTPUT_DIR,
    "table1_final_600cases.csv",
)

table1.reset_index().to_csv(
    TABLE1_PATH,
    index=False,
    encoding="utf-8-sig",
)

print("\nSaved:")
print(TABLE1_PATH)

In [ ]:
# ============================================================
# TABLE 1 — FULL TOPIC BREAKDOWN
# NLI + STANCE + RUNTIME FOR EACH TOPIC AND OVERALL
# ============================================================

import pandas as pd

df = pd.read_csv(RESULTS_PATH)

METHOD_LABELS = {
    "zero_shot_gpt_no_dataset": "Zero-Shot",
    "few_shot_task_prompting_no_dataset": "Few-Shot",
    "SPR_retrieval_only_raw_700k": "SPR",
    "partner_matters_persona_fusion_response_selection": "Partner Matters",
    "coo_style_usefulness_ranking_raw_700k": "COO",
    "personadb_offline_hierarchical_refinement": "Persona-DB",
    "ours_stance_nli_gpt_final_repository": "Our Framework",
}

METHOD_ORDER = list(METHOD_LABELS.keys())


# ------------------------------------------------------------
# Topic-specific results
# ------------------------------------------------------------

topic_summary = (
    df.groupby(["method", "topic"])
      .agg(
          NLI=("nli_contradiction", "mean"),
          Stance=("opposite_stance_accuracy", "mean"),
          Runtime=("runtime_seconds", "mean"),
      )
      .reset_index()
)


# ------------------------------------------------------------
# Overall results
# ------------------------------------------------------------

overall_summary = (
    df.groupby("method")
      .agg(
          NLI=("nli_contradiction", "mean"),
          Stance=("opposite_stance_accuracy", "mean"),
          Runtime=("runtime_seconds", "mean"),
      )
      .reset_index()
)

overall_summary["topic"] = "overall"


# ------------------------------------------------------------
# Combine
# ------------------------------------------------------------

summary = pd.concat(
    [topic_summary, overall_summary],
    ignore_index=True
)


# ------------------------------------------------------------
# Pivot into paper-style table
# ------------------------------------------------------------

table = summary.pivot(
    index="method",
    columns="topic",
    values=["NLI", "Stance", "Runtime"]
)


# Desired topic order
topics = [
    "blm",
    "guns",
    "samesex",
    "overall",
]

metrics = [
    "NLI",
    "Stance",
    "Runtime",
]


# Rebuild columns in:
# BLM NLI / Stance / Runtime /
# Guns NLI / Stance / Runtime / ...
final = pd.DataFrame(index=METHOD_ORDER)

for topic in topics:
    for metric in metrics:

        final[
            f"{topic}_{metric}"
        ] = table[
            (metric, topic)
        ].reindex(METHOD_ORDER)


# Friendly method names
final.index = [
    METHOD_LABELS[m]
    for m in final.index
]

final.index.name = "Method"


# ------------------------------------------------------------
# Round for paper
# ------------------------------------------------------------

paper = final.copy()

for col in paper.columns:

    if col.endswith("_Runtime"):
        paper[col] = paper[col].round(2)

    else:
        paper[col] = paper[col].round(3)


print("=" * 100)
print("FULL TABLE 1 — TOPIC + OVERALL")
print("=" * 100)

display(paper)

In [ ]:
# ============================================================
# STATISTICAL TESTS — OUR METHOD VS. ALL BASELINES
# 600 PAIRED TEST CASES
#
# NLI:
#   - Paired Wilcoxon signed-rank test
#   - Mean paired difference (Ours - Baseline)
#   - Bootstrap 95% CI
#   - Rank-biserial correlation
#   - Holm correction across 6 baseline comparisons
#
# Opposite-stance accuracy:
#   - Exact McNemar test
#   - Accuracy difference (Ours - Baseline)
#   - Holm correction across 6 baseline comparisons
# ============================================================

import os
import numpy as np
import pandas as pd

from scipy.stats import wilcoxon, binomtest
from statsmodels.stats.multitest import multipletests


# ------------------------------------------------------------
# 1. Load final 600-case results
# ------------------------------------------------------------

df = pd.read_csv(RESULTS_PATH)

print("=" * 75)
print("BASELINE STATISTICAL ANALYSIS — 600 PAIRED CASES")
print("=" * 75)

print("Loaded rows:", len(df))
print("Methods:", df["method"].nunique())


# ------------------------------------------------------------
# 2. Configuration
# ------------------------------------------------------------

OURS = "ours_stance_nli_gpt_final_repository"

BASELINES = [
    "zero_shot_gpt_no_dataset",
    "few_shot_task_prompting_no_dataset",
    "SPR_retrieval_only_raw_700k",
    "partner_matters_persona_fusion_response_selection",
    "coo_style_usefulness_ranking_raw_700k",
    "personadb_offline_hierarchical_refinement",
]

METHOD_LABELS_STATS = {
    "zero_shot_gpt_no_dataset":
        "Zero-Shot Opposite-Stance Prompting",

    "few_shot_task_prompting_no_dataset":
        "Few-Shot Opposite-Stance Prompting",

    "SPR_retrieval_only_raw_700k":
        "Semantic Persona Retrieval (SPR)",

    "partner_matters_persona_fusion_response_selection":
        "Persona Fusion Retrieval (Partner Matters)",

    "coo_style_usefulness_ranking_raw_700k":
        "Usefulness-Aware Persona Selection (COO)",

    "personadb_offline_hierarchical_refinement":
        "Hierarchical Persona Refinement (Persona-DB)",

    OURS:
        "Contradiction-Aware Persona Retrieval (Ours)",
}


# The three fields uniquely identify one test case.
CASE_COLS = [
    "topic",
    "user_opinion",
    "expected_user_stance",
]


# ------------------------------------------------------------
# 3. Basic QA
# ------------------------------------------------------------

required_cols = CASE_COLS + [
    "method",
    "nli_contradiction",
    "opposite_stance_accuracy",
]

missing = [c for c in required_cols if c not in df.columns]

if missing:
    raise ValueError(
        f"Missing required columns: {missing}"
    )

assert OURS in set(df["method"]), \
    "Our method is missing."

for baseline in BASELINES:
    assert baseline in set(df["method"]), \
        f"Missing baseline: {baseline}"


# Check duplicate experiment rows.
duplicates = df.duplicated(
    subset=CASE_COLS + ["method"]
).sum()

print("Duplicate experiment rows:", duplicates)

assert duplicates == 0, (
    f"Found {duplicates} duplicate experiment rows."
)


# ------------------------------------------------------------
# 4. Create stable case ID
# ------------------------------------------------------------

df = df.copy()

df["case_id"] = (
    df["topic"].astype(str)
    + "|||"
    + df["expected_user_stance"].astype(str)
    + "|||"
    + df["user_opinion"].astype(str)
)


# ------------------------------------------------------------
# 5. Helper — pair our method with one baseline
# ------------------------------------------------------------

def make_paired_df(metric, baseline):

    ours_df = (
        df[df["method"] == OURS]
        [["case_id", metric]]
        .rename(columns={metric: "ours"})
    )

    baseline_df = (
        df[df["method"] == baseline]
        [["case_id", metric]]
        .rename(columns={metric: "baseline"})
    )

    paired = ours_df.merge(
        baseline_df,
        on="case_id",
        how="inner",
        validate="one_to_one",
    )

    paired["ours"] = pd.to_numeric(
        paired["ours"],
        errors="coerce",
    )

    paired["baseline"] = pd.to_numeric(
        paired["baseline"],
        errors="coerce",
    )

    paired = paired.dropna(
        subset=["ours", "baseline"]
    ).reset_index(drop=True)

    return paired


# ------------------------------------------------------------
# 6. Helper — bootstrap CI for paired MEAN difference
# ------------------------------------------------------------

def bootstrap_mean_difference_ci(
    differences,
    n_boot=10000,
    seed=42,
):

    differences = np.asarray(
        differences,
        dtype=float,
    )

    rng = np.random.default_rng(seed)

    n = len(differences)

    boot_means = np.empty(
        n_boot,
        dtype=float,
    )

    for i in range(n_boot):

        sample = rng.choice(
            differences,
            size=n,
            replace=True,
        )

        boot_means[i] = np.mean(sample)

    low, high = np.percentile(
        boot_means,
        [2.5, 97.5],
    )

    return float(low), float(high)


# ------------------------------------------------------------
# 7. Helper — paired rank-biserial correlation
#
# Positive value:
# Ours tends to have higher NLI than baseline.
#
# Negative value:
# Baseline tends to have higher NLI than ours.
# ------------------------------------------------------------

def paired_rank_biserial(differences):

    differences = np.asarray(
        differences,
        dtype=float,
    )

    # Wilcoxon ignores zero differences.
    nonzero = differences[differences != 0]

    if len(nonzero) == 0:
        return 0.0

    abs_diff = np.abs(nonzero)

    ranks = pd.Series(abs_diff).rank(
        method="average"
    ).to_numpy()

    positive_rank_sum = ranks[
        nonzero > 0
    ].sum()

    negative_rank_sum = ranks[
        nonzero < 0
    ].sum()

    total = (
        positive_rank_sum
        + negative_rank_sum
    )

    if total == 0:
        return 0.0

    return float(
        (positive_rank_sum - negative_rank_sum)
        / total
    )


# ============================================================
# PART A — NLI CONTRADICTION
# ============================================================

print("\n" + "=" * 75)
print("PART A — NLI CONTRADICTION")
print("=" * 75)

nli_rows = []


for baseline in BASELINES:

    paired = make_paired_df(
        metric="nli_contradiction",
        baseline=baseline,
    )

    if len(paired) != 600:
        print(
            f"WARNING: {baseline} has "
            f"{len(paired)} valid NLI pairs, not 600."
        )

    ours = paired["ours"].to_numpy()
    base = paired["baseline"].to_numpy()

    diff = ours - base


    # --------------------------------------------------------
    # Wilcoxon signed-rank test
    # H1: distributions differ (two-sided)
    # --------------------------------------------------------

    if np.allclose(diff, 0):

        W = 0.0
        p_raw = 1.0

    else:

        test = wilcoxon(
            ours,
            base,
            alternative="two-sided",
            zero_method="wilcox",
            method="auto",
        )

        W = float(test.statistic)
        p_raw = float(test.pvalue)


    # --------------------------------------------------------
    # Bootstrap 95% CI
    # --------------------------------------------------------

    ci_low, ci_high = (
        bootstrap_mean_difference_ci(
            diff,
            n_boot=10000,
            seed=42,
        )
    )


    # --------------------------------------------------------
    # Rank-biserial effect size
    # --------------------------------------------------------

    r_rb = paired_rank_biserial(diff)


    nli_rows.append({

        "baseline":
            baseline,

        "baseline_label":
            METHOD_LABELS_STATS[baseline],

        "n_pairs":
            len(paired),

        "ours_mean_nli":
            float(np.mean(ours)),

        "baseline_mean_nli":
            float(np.mean(base)),

        "mean_difference_ours_minus_baseline":
            float(np.mean(diff)),

        "bootstrap_95CI_low":
            ci_low,

        "bootstrap_95CI_high":
            ci_high,

        "median_paired_difference":
            float(np.median(diff)),

        "wilcoxon_W":
            W,

        "p_raw":
            p_raw,

        "rank_biserial":
            r_rb,
    })


nli_stats = pd.DataFrame(nli_rows)


# ------------------------------------------------------------
# Holm correction — NLI
# ------------------------------------------------------------

reject, p_holm, _, _ = multipletests(
    nli_stats["p_raw"].to_numpy(),
    alpha=0.05,
    method="holm",
)

nli_stats["p_holm"] = p_holm

nli_stats["significant_after_holm"] = reject


# Sort by NLI difference.
nli_stats = (
    nli_stats
    .sort_values(
        "mean_difference_ours_minus_baseline",
        ascending=False,
    )
    .reset_index(drop=True)
)


print("\nNLI — Ours vs. each baseline:\n")

display(
    nli_stats[
        [
            "baseline_label",
            "n_pairs",
            "ours_mean_nli",
            "baseline_mean_nli",
            "mean_difference_ours_minus_baseline",
            "bootstrap_95CI_low",
            "bootstrap_95CI_high",
            "wilcoxon_W",
            "p_raw",
            "p_holm",
            "rank_biserial",
            "significant_after_holm",
        ]
    ].round(6)
)


# ============================================================
# PART B — OPPOSITE-STANCE ACCURACY
# ============================================================

print("\n" + "=" * 75)
print("PART B — OPPOSITE-STANCE ACCURACY")
print("=" * 75)

stance_rows = []


for baseline in BASELINES:

    paired = make_paired_df(
        metric="opposite_stance_accuracy",
        baseline=baseline,
    )

    if len(paired) != 600:
        print(
            f"WARNING: {baseline} has "
            f"{len(paired)} valid stance pairs, not 600."
        )


    # Convert explicitly to binary.
    ours = (
        paired["ours"]
        .round()
        .astype(int)
        .to_numpy()
    )

    base = (
        paired["baseline"]
        .round()
        .astype(int)
        .to_numpy()
    )


    # --------------------------------------------------------
    # McNemar discordant pairs
    #
    # b = ours correct, baseline wrong
    # c = ours wrong, baseline correct
    # --------------------------------------------------------

    b = int(
        np.sum(
            (ours == 1)
            & (base == 0)
        )
    )

    c = int(
        np.sum(
            (ours == 0)
            & (base == 1)
        )
    )

    discordant = b + c


    # Exact McNemar is equivalent to an exact
    # two-sided binomial test on discordant pairs.
    if discordant == 0:

        p_raw = 1.0

    else:

        p_raw = float(
            binomtest(
                k=min(b, c),
                n=discordant,
                p=0.5,
                alternative="two-sided",
            ).pvalue
        )


    ours_acc = float(np.mean(ours))
    baseline_acc = float(np.mean(base))


    stance_rows.append({

        "baseline":
            baseline,

        "baseline_label":
            METHOD_LABELS_STATS[baseline],

        "n_pairs":
            len(paired),

        "ours_accuracy":
            ours_acc,

        "baseline_accuracy":
            baseline_acc,

        "accuracy_difference_ours_minus_baseline":
            ours_acc - baseline_acc,

        "ours_correct_baseline_wrong":
            b,

        "ours_wrong_baseline_correct":
            c,

        "discordant_pairs":
            discordant,

        "mcnemar_exact_p_raw":
            p_raw,
    })


stance_stats = pd.DataFrame(
    stance_rows
)


# ------------------------------------------------------------
# Holm correction — stance
# ------------------------------------------------------------

reject, p_holm, _, _ = multipletests(
    stance_stats[
        "mcnemar_exact_p_raw"
    ].to_numpy(),
    alpha=0.05,
    method="holm",
)

stance_stats["p_holm"] = p_holm

stance_stats[
    "significant_after_holm"
] = reject


stance_stats = (
    stance_stats
    .sort_values(
        "accuracy_difference_ours_minus_baseline",
        ascending=False,
    )
    .reset_index(drop=True)
)


print(
    "\nOpposite-stance accuracy — "
    "Ours vs. each baseline:\n"
)

display(
    stance_stats[
        [
            "baseline_label",
            "n_pairs",
            "ours_accuracy",
            "baseline_accuracy",
            "accuracy_difference_ours_minus_baseline",
            "ours_correct_baseline_wrong",
            "ours_wrong_baseline_correct",
            "discordant_pairs",
            "mcnemar_exact_p_raw",
            "p_holm",
            "significant_after_holm",
        ]
    ].round(6)
)


# ============================================================
# PART C — SAVE RESULTS
# ============================================================

NLI_STATS_PATH = os.path.join(
    OUTPUT_DIR,
    "baseline_nli_statistical_tests_600cases.csv",
)

STANCE_STATS_PATH = os.path.join(
    OUTPUT_DIR,
    "baseline_stance_statistical_tests_600cases.csv",
)


nli_stats.to_csv(
    NLI_STATS_PATH,
    index=False,
    encoding="utf-8-sig",
)

stance_stats.to_csv(
    STANCE_STATS_PATH,
    index=False,
    encoding="utf-8-sig",
)


print("\n" + "=" * 75)
print("SAVED")
print("=" * 75)

print("NLI statistics:")
print(NLI_STATS_PATH)

print("\nStance statistics:")
print(STANCE_STATS_PATH)


# ============================================================
# PART D — COMPACT INTERPRETATION CHECK
# ============================================================

print("\n" + "=" * 75)
print("COMPACT SIGNIFICANCE SUMMARY")
print("=" * 75)


print("\nNLI:")

for _, row in nli_stats.iterrows():

    direction = (
        "OURS HIGHER"
        if row[
            "mean_difference_ours_minus_baseline"
        ] > 0
        else "BASELINE HIGHER"
    )

    sig = (
        "SIGNIFICANT"
        if row["significant_after_holm"]
        else "NOT SIGNIFICANT"
    )

    print(
        f"{row['baseline_label']}: "
        f"{direction} | "
        f"Δ={row['mean_difference_ours_minus_baseline']:.4f} | "
        f"Holm p={row['p_holm']:.6g} | "
        f"{sig}"
    )


print("\nOpposite-stance accuracy:")

for _, row in stance_stats.iterrows():

    direction = (
        "OURS HIGHER"
        if row[
            "accuracy_difference_ours_minus_baseline"
        ] > 0
        else (
            "BASELINE HIGHER"
            if row[
                "accuracy_difference_ours_minus_baseline"
            ] < 0
            else "EQUAL"
        )
    )

    sig = (
        "SIGNIFICANT"
        if row["significant_after_holm"]
        else "NOT SIGNIFICANT"
    )

    print(
        f"{row['baseline_label']}: "
        f"{direction} | "
        f"Δacc={row['accuracy_difference_ours_minus_baseline']:.4f} | "
        f"Holm p={row['p_holm']:.6g} | "
        f"{sig}"
    )


print("\n✅ Statistical analysis complete.")